# ESMM Paper Reproduction — AliCCP Dataset (Colab)

This notebook reproduces experiments from "Entire Space Multi-Task Model: An Effective Approach for Estimating Post-Click Conversion Rate" (SIGIR 2018).

**Paper key idea:** Multi-task learning with shared embeddings. Two towers (CTR + CVR), pCTCVR = pCTR × pCVR. Loss = BCE(click, pCTR) + BCE(click&purchase, pCTCVR) on ALL impressions.

**Data:** Ali-CCP dataset from https://tianchi.aliyun.com/dataset/408. Upload `sample_train.tar` and `sample_test.tar` into DATA_DIR.

**Colab SSH:** use your current `cloudflared` hostname. **If SSH drops mid-run:** reconnect and re-run — caches under `DATA_DIR` skip finished work.

**Git / scp:** The repo-sync cell below **skips `git fetch` / `reset --hard` by default** so a notebook copied with **`scp` + `papermill`** is not replaced by GitHub `main`. To pull latest from GitHub in the Colab UI, set `SKIP_GIT_REPO_SYNC = False` in that cell or run with env **`FORCE_GIT_SYNC=1`**.

**Re-run a round or wipe partial files:** change flags in the **Config** cell (`CLEAN_ROUND_RESULT_JSON`, `CLEAN_R4_NORMALIZED_PARQUET`, `CLEAN_R4_VOCAB_CACHE`). Cleanup uses Python `os.remove` on **Drive** inside the notebook — do not rely on the agent running shell `rm` (that triggers editor approval dialogs).

**K-only runs:** set `RUN_ROUND4_K_ONLY = True` in the Config cell to skip all experiments except Round 4 Experiment K (uses `round_4_k_only_results.json`).


In [ ]:
import os
if os.path.ismount('/content/drive'):
    print('Drive already mounted.')
else:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception:
        print('Skipping drive mount (not in Colab UI or drive unavailable).')

In [ ]:
import os
if not os.path.exists('/content/drive'):
    print('Warning: Drive not mounted, some paths may not work.')
WORK_DIR = '/content/drive/MyDrive/colab/recsys_playground'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

In [ ]:
import os, subprocess, shutil
try:
    import google.colab
    IN_COLAB = True
except Exception:
    IN_COLAB = False

repo_url = 'https://github.com/allyoushawn/recsys_playground.git'
repo_dir = 'recsys_playground'
branch_name = 'main'

# Default True: do NOT git fetch/reset — required for scp + papermill (see intro markdown).
# Colab UI: set False here, or export FORCE_GIT_SYNC=1 before Run All, to track GitHub main.
SKIP_GIT_REPO_SYNC = True
if os.environ.get('FORCE_GIT_SYNC', '').strip().lower() in ('1', 'true', 'yes'):
    SKIP_GIT_REPO_SYNC = False

if IN_COLAB:
    git_marker = os.path.join(repo_dir, '.git')
    if SKIP_GIT_REPO_SYNC:
        if os.path.isdir(repo_dir):
            os.chdir(repo_dir)
            print('[SKIP_GIT_REPO_SYNC] Skipping git fetch/reset — using Drive copy (scp/papermill safe).')
        else:
            print('[SKIP_GIT_REPO_SYNC] Repo dir missing; cloning once (no reset).')
            subprocess.run(['git', 'clone', repo_url], check=True)
            os.chdir(repo_dir)
            subprocess.run(['git', 'checkout', branch_name], check=False)
    elif os.path.isdir(repo_dir) and os.path.isdir(git_marker):
        os.chdir(repo_dir)
        subprocess.run(['git', 'fetch', '--all'], check=True)
        subprocess.run(['git', 'checkout', branch_name], check=False)
        subprocess.run(['git', 'reset', '--hard', f'origin/{branch_name}'], check=False)
    else:
        if os.path.exists(repo_dir):
            shutil.rmtree(repo_dir)
        subprocess.run(['git', 'clone', repo_url], check=True)
        os.chdir(repo_dir)
        subprocess.run(['git', 'fetch', '--all'], check=True)
        subprocess.run(['git', 'checkout', branch_name], check=False)

In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'torch', 'pandas', 'numpy', 'scikit-learn',
                'matplotlib', 'seaborn', 'requests', 'tqdm', 'joblib',
                'pyarrow', 'psutil'], check=True)

In [ ]:
import os

PROJECT_NAME = 'esmm_experiment'
DATA_DIR = '/content/drive/MyDrive/colab/data/ali_ccp'
# Processed + experiment caches under DATA_DIR (clear subfolders; avoids re-parsing / re-training).
PROCESSED_PARSED_DIR = os.path.join(DATA_DIR, 'processed_esmm_parsed_samples')
PROCESSED_FULL_DIR = os.path.join(DATA_DIR, 'processed_esmm_full_parquet')
ROUND_RESULTS_DIR = os.path.join(DATA_DIR, 'esmm_round_training_cache')
for _d in (PROCESSED_PARSED_DIR, PROCESSED_FULL_DIR, ROUND_RESULTS_DIR):
    os.makedirs(_d, exist_ok=True)
print('\n--- Processed outputs (all under DATA_DIR; safe to keep on Drive) ---')
print(f'  {PROCESSED_PARSED_DIR}/')
print('      parsed_train_rows_<N>.parquet / parsed_test_rows_<N>.parquet  (5M, 15M, ...)')
print(f'  {PROCESSED_FULL_DIR}/')
print('      parsed_train_rows_full.parquet / parsed_test_rows_full.parquet  (one-time parse)')
print('      r4_filtered_sparse_vocab.pkl  (freq-filter vocabs; skip scans if train file unchanged)')
print(f'  {ROUND_RESULTS_DIR}/')
print('      round_1_results.json … round_5_results.json  (delete a file to re-run that round)')
print('---\n')

# --- Notebook-side Drive cleanup (Colab only): use these instead of local shell `rm` / SSH rm (avoids Cursor approval prompts on the agent machine)
R4_NORM_TRAIN = os.path.join(PROCESSED_FULL_DIR, 'r4_norm_train.parquet')
R4_NORM_TEST = os.path.join(PROCESSED_FULL_DIR, 'r4_norm_test.parquet')
# Frequency-filtered sparse vocabs (Round 4 / 5 Parquet scans); rebuilt if train Parquet or settings change.
R4_FILTERED_VOCAB_CACHE = os.path.join(PROCESSED_FULL_DIR, 'r4_filtered_sparse_vocab.pkl')
# Set round numbers to delete esmm_round_training_cache/round_{n}_results.json before that round runs.
CLEAN_ROUND_RESULT_JSON = []  # e.g. [4] to force Round 4 to re-execute
# Drop Round-4 normalized spill files if a partial/corrupt Parquet was written.
CLEAN_R4_NORMALIZED_PARQUET = False
# Remove vocab pickle to force full Parquet vocab scan on next run.
CLEAN_R4_VOCAB_CACHE = False
# If True, ignore pickle and rescan (overwrites cache at end).
FORCE_REBUILD_R4_VOCAB = False

# When True: skip sample CSV/Parquet load, EDA, Rounds 1–3, experiment J, and Rounds 5–6.
# Runs only Round 4 Experiment K (full-split Parquet prep + ESMM row-group training).
# Cache file: round_4_k_only_results.json (separate from round_4_results.json).
RUN_ROUND4_K_ONLY = True

# --- Round 6 MTL: if True, skip that architecture leg (not trained). ---
SKIP_ROUND6_K_REF = False
SKIP_ROUND6_SHARED_BOTTOM = False
SKIP_ROUND6_MMOE = False
SKIP_ROUND6_PLE = False
# PLE-only Colab: set env ESMM_TUNNEL_HOST to your trycloudflare hostname (same as ssh), e.g.
#   ESMM_TUNNEL_HOST=served-melissa-subjective-acrobat.trycloudflare.com papermill ...
COLAB_TUNNEL_HOST_FOR_PLE_ONLY_RUN = 'served-melissa-subjective-acrobat.trycloudflare.com'
_ESMM_TUNNEL = os.environ.get('ESMM_TUNNEL_HOST', '').strip()

if _ESMM_TUNNEL == COLAB_TUNNEL_HOST_FOR_PLE_ONLY_RUN:
    RUN_ROUND4_K_ONLY = False
    SKIP_ROUND6_K_REF = True
    SKIP_ROUND6_SHARED_BOTTOM = True
    SKIP_ROUND6_MMOE = True
    SKIP_ROUND6_PLE = False
    print(
        f'[ESMM_TUNNEL_HOST] Matched {COLAB_TUNNEL_HOST_FOR_PLE_ONLY_RUN!r} → PLE-only (R6); '
        'RUN_ROUND4_K_ONLY=False (requires existing round_4_results.json + R4 Parquet on Drive).'
    )

def _drive_remove(path, desc):
    if os.path.isfile(path):
        os.remove(path)
        print(f'[cleanup] Removed {desc}: {path}')

for _rn in CLEAN_ROUND_RESULT_JSON:
    _p = os.path.join(ROUND_RESULTS_DIR, f'round_{_rn}_results.json')
    _drive_remove(_p, f'round_{_rn}_results.json')

if CLEAN_R4_NORMALIZED_PARQUET:
    _drive_remove(R4_NORM_TRAIN, 'r4_norm_train.parquet')
    _drive_remove(R4_NORM_TEST, 'r4_norm_test.parquet')

if CLEAN_R4_VOCAB_CACHE:
    _drive_remove(R4_FILTERED_VOCAB_CACHE, 'r4_filtered_sparse_vocab.pkl')

# RAM-bounded streaming (Round 4 full split): tune down if Colab still OOMs.
STREAM_PARSE_CHUNK_ROWS = 500_000
VOCAB_SCAN_ROWS_PER_BATCH = 200_000
NORM_STREAM_BATCH_ROWS = 500_000
EVAL_TEST_BATCH_ROWS = 500_000

SAMPLE_SIZE = 5_000_000
RANDOM_STATE = 42
EMBED_DIM = 18

# Experiment K (ESMM Parquet row-groups): early-stop caps. None = unlimited (default full 5-epoch K unchanged).
K_EARLY_STOP_MAX_WALL_SECONDS = None  # e.g. 900 for ~15 min dev cap on Experiment K
K_EARLY_STOP_MAX_OPTIMIZER_STEPS = None
K_EARLY_STOP_MAX_BATCHES_PER_EPOCH = None
K_EARLY_STOP_MAX_ROW_GROUPS_PER_EPOCH = None

SPARSE_COLS = ['101', '121', '122', '124', '125', '126', '127', '128', '129',
               '205', '206', '207', '210', '216', '508', '509', '702', '853',
               '301', '109_14', '110_14', '127_14', '150_14']
DENSE_COLS = ['109_14', '110_14', '127_14', '150_14', '508', '509', '702', '853']
DENSE_FEAT_COLS = ['D' + c for c in DENSE_COLS]

print(f'Config: SAMPLE_SIZE={SAMPLE_SIZE:,}, EMBED_DIM={EMBED_DIM}')
print(f'RUN_ROUND4_K_ONLY={RUN_ROUND4_K_ONLY}')
print(f'ESMM_TUNNEL_HOST(environ)={_ESMM_TUNNEL!r}')
print(
    f'SKIP MTL: R6[K_ref={SKIP_ROUND6_K_REF} SharedBottom={SKIP_ROUND6_SHARED_BOTTOM} '
    f'MMoE={SKIP_ROUND6_MMOE} PLE={SKIP_ROUND6_PLE}]'
)
print(f'Sparse features: {len(SPARSE_COLS)}, Dense features: {len(DENSE_COLS)}')


In [ ]:
import os
import tarfile
import gc
import pickle
import numpy as np
import pandas as pd
from tqdm import tqdm

os.makedirs(DATA_DIR, exist_ok=True)

SAMPLE_TRAIN_TAR = 'sample_train.tar'
SAMPLE_TEST_TAR = 'sample_test.tar'
TRAIN_CSV = 'ali_ccp_train.csv'
VAL_CSV = 'ali_ccp_val.csv'
TEST_CSV = 'ali_ccp_test.csv'
SINGLE_CSV = 'ali_ccp.csv'
SAMPLE_SKELETON_TRAIN = 'sample_skeleton_train.csv'
SAMPLE_SKELETON_TEST = 'sample_skeleton_test.csv'
COMMON_FEATURES_TRAIN = 'common_features_train.csv'
COMMON_FEATURES_TEST = 'common_features_test.csv'

USES_COLS = SPARSE_COLS + DENSE_FEAT_COLS

def _sample_tag_for_cache(sample_size):
    return 'full' if sample_size is None else str(sample_size)

def load_or_parse_ali_ccp(data_dir, sample_size, processed_dir):
    """Load Parquet from processed_dir if present; else parse raw Tianchi CSVs and save."""
    os.makedirs(processed_dir, exist_ok=True)
    tag = _sample_tag_for_cache(sample_size)
    p_train = os.path.join(processed_dir, f'parsed_train_rows_{tag}.parquet')
    p_test = os.path.join(processed_dir, f'parsed_test_rows_{tag}.parquet')
    # Reuse legacy Parquet from older notebook layout (…/data/esmm_cache/) if present.
    if tag == 'full' and not (os.path.isfile(p_train) and os.path.isfile(p_test)):
        _legacy = os.path.join(os.path.dirname(os.path.abspath(data_dir)), 'esmm_cache')
        _lt, _le = os.path.join(_legacy, 'ali_ccp_full_train.parquet'), os.path.join(_legacy, 'ali_ccp_full_test.parquet')
        if os.path.isfile(_lt) and os.path.isfile(_le):
            import shutil
            print(f'Copying full split from legacy {_legacy} -> {processed_dir}')
            shutil.copy2(_lt, p_train)
            shutil.copy2(_le, p_test)
    if os.path.isfile(p_train) and os.path.isfile(p_test):
        print(f'Loading cached parsed AliCCP (sample={tag}) from {processed_dir}')
        if sample_size is None:
            print('  Full split: not calling pd.read_parquet (avoids ~40GB+ RAM). Use Parquet paths in Round 4.')
            return None, None
        return pd.read_parquet(p_train), pd.read_parquet(p_test)
    print(f'No Parquet cache for sample={tag}; parsing raw Tianchi files (slow)...')
    if sample_size is None:
        parse_raw_ali_ccp_streaming_writes(data_dir, p_train, p_test, sample_size=None)
        print('  Full Parquet written. Returning (None, None) to avoid read_parquet OOM.')
        return None, None
    df_tr, df_te = parse_raw_ali_ccp(data_dir, sample_size=sample_size)
    if df_tr is not None and len(df_tr) > 0:
        print(f'Writing Parquet cache to {processed_dir} (snappy compression)...')
        df_tr.to_parquet(p_train, index=False, compression='snappy')
        df_te.to_parquet(p_test, index=False, compression='snappy')
    return df_tr, df_te

def _find_file_recursive(root, filename):
    direct = os.path.join(root, filename)
    if os.path.isfile(direct):
        return direct
    for dirpath, _, filenames in os.walk(root):
        if filename in filenames:
            return os.path.join(dirpath, filename)
    return None

def _parse_feat_str(feat_str, sparse_cols, dense_cols):
    feat_dict = {}
    for fstr in feat_str.split('\x01'):
        if '\x02' not in fstr or '\x03' not in fstr:
            continue
        parts = fstr.split('\x02', 1)
        filed = parts[0]
        feat_val = parts[1]
        if '\x03' in feat_val:
            feat, val = feat_val.split('\x03', 1)
            if filed in sparse_cols:
                feat_dict[filed] = feat
            if filed in dense_cols:
                feat_dict['D' + filed] = val
    return feat_dict

def parse_raw_ali_ccp(data_dir, sample_size=None):
    common_train_path = _find_file_recursive(data_dir, COMMON_FEATURES_TRAIN)
    common_test_path = _find_file_recursive(data_dir, COMMON_FEATURES_TEST)
    skeleton_train_path = _find_file_recursive(data_dir, SAMPLE_SKELETON_TRAIN)
    skeleton_test_path = _find_file_recursive(data_dir, SAMPLE_SKELETON_TEST)
    if not all([common_train_path, common_test_path, skeleton_train_path, skeleton_test_path]):
        return None, None

    test_limit = (sample_size // 4) if sample_size else None

    needed_ids = set()
    for path, limit, mode in [
        (skeleton_train_path, sample_size, 'train'),
        (skeleton_test_path, test_limit, 'test'),
    ]:
        with open(path, 'r') as f:
            for i, line in enumerate(tqdm(f, desc=f'scan_skeleton_{mode}', leave=False)):
                if limit and i >= limit:
                    break
                parts = line.strip().split(',')
                if len(parts) >= 4:
                    needed_ids.add(parts[3])
    print(f'Unique common-feature IDs needed: {len(needed_ids):,}')

    common_feat = {}
    for path, mode in [(common_train_path, 'train'), (common_test_path, 'test')]:
        with open(path, 'r') as f:
            for line in tqdm(f, desc=f'common_features_{mode}', leave=False):
                parts = line.strip().split(',')
                if len(parts) < 3:
                    continue
                if parts[0] not in needed_ids:
                    continue
                feat_dict = _parse_feat_str(parts[2], SPARSE_COLS, DENSE_COLS)
                common_feat[parts[0]] = feat_dict
    print(f'Loaded {len(common_feat):,} common-feature entries')

    rows_train, rows_test = [], []
    for path, rows_out, limit, mode in [
        (skeleton_train_path, rows_train, sample_size, 'train'),
        (skeleton_test_path, rows_test, test_limit, 'test'),
    ]:
        with open(path, 'r') as f:
            for i, line in enumerate(tqdm(f, desc=f'sample_skeleton_{mode}', leave=False)):
                if limit and i >= limit:
                    break
                parts = line.strip().split(',')
                if len(parts) < 6:
                    continue
                click, purchase = parts[1], parts[2]
                if click == '0' and purchase == '1':
                    continue
                feat_dict = _parse_feat_str(parts[5], SPARSE_COLS, DENSE_COLS)
                feat_dict.update(common_feat.get(parts[3], {}))
                row = {'click': click, 'purchase': purchase}
                for k in USES_COLS:
                    row[k] = feat_dict.get(k, '0')
                rows_out.append(row)
    df_train = pd.DataFrame(rows_train)
    df_test = pd.DataFrame(rows_test)
    return df_train, df_test


def parse_raw_ali_ccp_streaming_writes(data_dir, p_train_out, p_test_out, sample_size=None, chunk_rows=None):
    """Write full (or sampled) train/test Parquet in chunks — avoids 42M-row Python list + giant DataFrame."""
    import pyarrow as pa
    import pyarrow.parquet as pq

    if chunk_rows is None:
        chunk_rows = STREAM_PARSE_CHUNK_ROWS

    common_train_path = _find_file_recursive(data_dir, COMMON_FEATURES_TRAIN)
    common_test_path = _find_file_recursive(data_dir, COMMON_FEATURES_TEST)
    skeleton_train_path = _find_file_recursive(data_dir, SAMPLE_SKELETON_TRAIN)
    skeleton_test_path = _find_file_recursive(data_dir, SAMPLE_SKELETON_TEST)
    if not all([common_train_path, common_test_path, skeleton_train_path, skeleton_test_path]):
        raise FileNotFoundError('Missing raw Tianchi CSVs under DATA_DIR')

    test_limit = (sample_size // 4) if sample_size else None

    needed_ids = set()
    for path, limit, mode in [
        (skeleton_train_path, sample_size, 'train'),
        (skeleton_test_path, test_limit, 'test'),
    ]:
        with open(path, 'r') as f:
            for i, line in enumerate(tqdm(f, desc=f'scan_skeleton_{mode}', leave=False)):
                if limit and i >= limit:
                    break
                parts = line.strip().split(',')
                if len(parts) >= 4:
                    needed_ids.add(parts[3])
    print(f'Unique common-feature IDs needed: {len(needed_ids):,}')

    common_feat = {}
    for path, mode in [(common_train_path, 'train'), (common_test_path, 'test')]:
        with open(path, 'r') as f:
            for line in tqdm(f, desc=f'common_features_{mode}', leave=False):
                parts = line.strip().split(',')
                if len(parts) < 3:
                    continue
                if parts[0] not in needed_ids:
                    continue
                common_feat[parts[0]] = _parse_feat_str(parts[2], SPARSE_COLS, DENSE_COLS)
    print(f'Loaded {len(common_feat):,} common-feature entries')

    def skeleton_rows(path, limit, desc):
        with open(path, 'r') as f:
            for i, line in enumerate(tqdm(f, desc=desc, leave=False)):
                if limit and i >= limit:
                    break
                parts = line.strip().split(',')
                if len(parts) < 6:
                    continue
                click, purchase = parts[1], parts[2]
                if click == '0' and purchase == '1':
                    continue
                fd = _parse_feat_str(parts[5], SPARSE_COLS, DENSE_COLS)
                fd.update(common_feat.get(parts[3], {}))
                row = {'click': int(click), 'purchase': int(purchase)}
                for k in USES_COLS:
                    row[k] = fd.get(k, '0')
                yield row

    def flush_write(buf, writer_holder, out_path, is_first_table):
        if not buf:
            return writer_holder, is_first_table
        df = pd.DataFrame(buf)
        buf.clear()
        table = pa.Table.from_pandas(df, preserve_index=False)
        del df
        if writer_holder[0] is None:
            writer_holder[0] = pq.ParquetWriter(out_path, table.schema, compression='snappy')
        writer_holder[0].write_table(table)
        del table
        gc.collect()
        return writer_holder, is_first_table

    def write_split(skel_path, limit, out_path, desc):
        buf, wh = [], [None]
        for row in skeleton_rows(skel_path, limit, desc):
            buf.append(row)
            if len(buf) >= chunk_rows:
                wh, _ = flush_write(buf, wh, out_path, False)
        flush_write(buf, wh, out_path, False)
        if wh[0] is not None:
            wh[0].close()
        gc.collect()
        print(f'Wrote {out_path}')

    write_split(skeleton_train_path, sample_size, p_train_out, 'stream_parse_train')
    del common_feat
    gc.collect()
    common_feat = {}
    for path, mode in [(common_train_path, 'train'), (common_test_path, 'test')]:
        with open(path, 'r') as f:
            for line in tqdm(f, desc=f'common_features_{mode}_re', leave=False):
                parts = line.strip().split(',')
                if len(parts) < 3 or parts[0] not in needed_ids:
                    continue
                common_feat[parts[0]] = _parse_feat_str(parts[2], SPARSE_COLS, DENSE_COLS)
    print(f'Reloaded {len(common_feat):,} common-feature entries for test split')
    write_split(skeleton_test_path, test_limit, p_test_out, 'stream_parse_test')
    del common_feat
    gc.collect()


def build_sparse_vocabs_filtered_parquet(parquet_path, sparse_cols, min_count=5):
    from collections import Counter
    import pyarrow.parquet as pq

    vocabs, cardinalities = {}, []
    pf = pq.ParquetFile(parquet_path)
    for col in sparse_cols:
        c = Counter()
        for batch in pf.iter_batches(batch_size=VOCAB_SCAN_ROWS_PER_BATCH, columns=[col]):
            for v in batch.column(0).to_pylist():
                c[str(v)] += 1
            del batch
        kept_vals = sorted([x for x, y in c.items() if y >= min_count], key=lambda x: -c[x])
        vocab = {v: i + 1 for i, v in enumerate(kept_vals)}
        vocabs[col] = vocab
        cardinalities.append(len(vocab))
        print(f'  {col}: {len(c)} unique, {len(kept_vals)} kept (>={min_count}), {len(c) - len(kept_vals)} filtered')
        del c
    return vocabs, cardinalities


def _try_load_filtered_vocab_cache(cache_path, parquet_path, sparse_cols, min_count):
    """Return (vocabs, cardinalities) if cache matches train Parquet + settings; else None."""
    import pyarrow.parquet as pq
    if not cache_path or not os.path.isfile(cache_path):
        return None
    ap = os.path.abspath(parquet_path)
    if not os.path.isfile(ap):
        return None
    try:
        with open(cache_path, 'rb') as f:
            payload = pickle.load(f)
    except Exception as e:
        print(f'[vocab cache] unreadable ({e}); rebuilding')
        return None
    meta, vocabs = payload.get('meta'), payload.get('vocabs')
    if not isinstance(meta, dict) or not isinstance(vocabs, dict):
        return None
    if meta.get('parquet_path') != ap:
        return None
    if list(meta.get('sparse_cols') or []) != list(sparse_cols):
        return None
    if int(meta.get('min_count', -1)) != int(min_count):
        return None
    cur_mtime = os.path.getmtime(ap)
    if meta.get('parquet_mtime') != cur_mtime:
        print('[vocab cache] train Parquet mtime changed; rebuilding vocabs')
        return None
    cur_rows = int(pq.ParquetFile(ap).metadata.num_rows)
    if int(meta.get('parquet_num_rows', -1)) != cur_rows:
        print('[vocab cache] train Parquet row count changed; rebuilding vocabs')
        return None
    cards = meta.get('cardinalities')
    if not cards or len(cards) != len(sparse_cols):
        return None
    for col in sparse_cols:
        if col not in vocabs:
            return None
    print(f'[vocab cache] loaded {cache_path} (skip {len(sparse_cols)} Parquet column scans)')
    return vocabs, list(cards)


def _save_filtered_vocab_cache(cache_path, vocabs, cardinalities, parquet_path, sparse_cols, min_count):
    import pyarrow.parquet as pq
    ap = os.path.abspath(parquet_path)
    os.makedirs(os.path.dirname(os.path.abspath(cache_path)) or '.', exist_ok=True)
    meta = {
        'version': 1,
        'parquet_path': ap,
        'parquet_mtime': os.path.getmtime(ap),
        'parquet_num_rows': int(pq.ParquetFile(ap).metadata.num_rows),
        'sparse_cols': list(sparse_cols),
        'min_count': int(min_count),
        'cardinalities': list(cardinalities),
    }
    tmp = cache_path + '.tmp'
    with open(tmp, 'wb') as f:
        pickle.dump({'meta': meta, 'vocabs': vocabs}, f, protocol=4)
    os.replace(tmp, cache_path)
    print(f'[vocab cache] wrote {cache_path}')


def load_or_build_sparse_vocabs_filtered_parquet(
    parquet_path, sparse_cols, min_count=5, cache_path=None, force_rebuild=False,
):
    """Same as build_sparse_vocabs_filtered_parquet but loads from disk when cache is valid."""
    if not force_rebuild and cache_path:
        loaded = _try_load_filtered_vocab_cache(cache_path, parquet_path, sparse_cols, min_count)
        if loaded is not None:
            return loaded
    vocabs, cards = build_sparse_vocabs_filtered_parquet(parquet_path, sparse_cols, min_count=min_count)
    if cache_path:
        _save_filtered_vocab_cache(cache_path, vocabs, cards, parquet_path, sparse_cols, min_count)
    return vocabs, cards


def stream_normalize_parquet(in_path, out_path, sparse_cols, dense_feat_cols):
    import pyarrow as pa
    import pyarrow.parquet as pq

    cols = sparse_cols + dense_feat_cols + ['click', 'purchase']
    pf = pq.ParquetFile(in_path)
    writer = None
    for batch in pf.iter_batches(batch_size=NORM_STREAM_BATCH_ROWS, columns=cols):
        df = batch.to_pandas()
        del batch
        for dc in dense_feat_cols:
            x = pd.to_numeric(df[dc], errors='coerce').fillna(0.0)
            df[dc] = np.log1p(np.abs(x)) * np.sign(x)
        df['click'] = df['click'].astype(int)
        df['purchase'] = df['purchase'].astype(int)
        table = pa.Table.from_pandas(df, preserve_index=False)
        del df
        if writer is None:
            writer = pq.ParquetWriter(out_path, table.schema, compression='snappy')
        writer.write_table(table)
        del table
        gc.collect()
    if writer is None:
        raise RuntimeError(f'No rows in {in_path}')
    writer.close()
    gc.collect()
    print(f'Normalized -> {out_path}')


def ensure_full_split_parquet_streaming(data_dir, processed_dir):
    """Paths to full train/test Parquet; streaming-parse if missing."""
    import shutil
    os.makedirs(processed_dir, exist_ok=True)
    p_train = os.path.join(processed_dir, 'parsed_train_rows_full.parquet')
    p_test = os.path.join(processed_dir, 'parsed_test_rows_full.parquet')
    if not (os.path.isfile(p_train) and os.path.isfile(p_test)):
        _legacy = os.path.join(os.path.dirname(os.path.abspath(data_dir)), 'esmm_cache')
        _lt, _le = os.path.join(_legacy, 'ali_ccp_full_train.parquet'), os.path.join(_legacy, 'ali_ccp_full_test.parquet')
        if os.path.isfile(_lt) and os.path.isfile(_le):
            print(f'Copying legacy full split -> {processed_dir}')
            shutil.copy2(_lt, p_train)
            shutil.copy2(_le, p_test)
    if os.path.isfile(p_train) and os.path.isfile(p_test):
        return p_train, p_test
    print('Streaming parse: raw Tianchi -> full Parquet (chunked)...')
    parse_raw_ali_ccp_streaming_writes(data_dir, p_train, p_test, sample_size=None)
    return p_train, p_test


def parquet_split_summary(train_path, test_path):
    import pyarrow as pa
    import pyarrow.parquet as pq
    import pyarrow.compute as pc

    pf_t, pf_e = pq.ParquetFile(train_path), pq.ParquetFile(test_path)
    print(f'Parquet rows: train={pf_t.metadata.num_rows:,}, test={pf_e.metadata.num_rows:,}')
    for name, pth in [('train', train_path), ('test', test_path)]:
        t = pq.read_table(pth, columns=['click', 'purchase'])
        clicks = int(pc.sum(pc.cast(t['click'], pa.int64())).as_py() or 0)
        conv = int(pc.sum(pc.cast(t['purchase'], pa.int64())).as_py() or 0)
        print(f'  {name}: clicks={clicks:,}, conversions={conv:,}')
        del t

summarize_split = parquet_split_summary

train_path = os.path.join(DATA_DIR, TRAIN_CSV)
val_path = os.path.join(DATA_DIR, VAL_CSV)
test_path = os.path.join(DATA_DIR, TEST_CSV)
single_path = os.path.join(DATA_DIR, SINGLE_CSV)
train_tar_path = os.path.join(DATA_DIR, SAMPLE_TRAIN_TAR)
test_tar_path = os.path.join(DATA_DIR, SAMPLE_TEST_TAR)

has_archives = os.path.isfile(train_tar_path) and os.path.isfile(test_tar_path)
needs_extract = has_archives and (
    not _find_file_recursive(DATA_DIR, SAMPLE_SKELETON_TRAIN) or
    not _find_file_recursive(DATA_DIR, COMMON_FEATURES_TRAIN)
)
if needs_extract:
    for arc in [SAMPLE_TRAIN_TAR, SAMPLE_TEST_TAR]:
        path = os.path.join(DATA_DIR, arc)
        if os.path.isfile(path):
            print(f'Extracting {arc}...')
            with tarfile.open(path, 'r:*') as tf:
                tf.extractall(DATA_DIR)
            print(f'Done.')

has_splits = os.path.exists(train_path) and os.path.exists(test_path)
has_single = os.path.exists(single_path)
has_raw = (
    _find_file_recursive(DATA_DIR, SAMPLE_SKELETON_TRAIN) is not None and
    _find_file_recursive(DATA_DIR, COMMON_FEATURES_TRAIN) is not None
)

if RUN_ROUND4_K_ONLY:
    print('[RUN_ROUND4_K_ONLY] Skipping SAMPLE_SIZE train/test DataFrame load (Experiment K uses full-split Parquet only).')
    df_train = None
    df_test = None
else:
    has_splits = os.path.exists(train_path) and os.path.exists(test_path)
    has_single = os.path.exists(single_path)
    has_raw = (
        _find_file_recursive(DATA_DIR, SAMPLE_SKELETON_TRAIN) is not None and
        _find_file_recursive(DATA_DIR, COMMON_FEATURES_TRAIN) is not None
    )

    if has_splits:
        print(f'Using preprocessed splits: {train_path}, {test_path}')
        df_train = pd.read_csv(train_path, nrows=SAMPLE_SIZE)
        df_test = pd.read_csv(test_path, nrows=(SAMPLE_SIZE // 4) if SAMPLE_SIZE else None)
    elif has_single:
        print(f'Using single CSV: {single_path}')
        from sklearn.model_selection import train_test_split
        df = pd.read_csv(single_path, nrows=SAMPLE_SIZE)
        df_train, df_test = train_test_split(df, test_size=0.2, random_state=RANDOM_STATE)
    elif has_raw:
        df_train, df_test = load_or_parse_ali_ccp(DATA_DIR, SAMPLE_SIZE, PROCESSED_PARSED_DIR)
        if df_train is None or len(df_train) == 0:
            raise FileNotFoundError('Raw parsing failed.')
    else:
        raise FileNotFoundError(
            f'No data found in {DATA_DIR}. Download from https://tianchi.aliyun.com/dataset/408'
        )

    df_train['click'] = df_train['click'].astype(int)
    df_train['purchase'] = df_train['purchase'].astype(int)
    df_test['click'] = df_test['click'].astype(int)
    df_test['purchase'] = df_test['purchase'].astype(int)
    print(f'Loaded: train {df_train.shape}, test {df_test.shape}')



In [ ]:
import numpy as np
import pandas as pd

if RUN_ROUND4_K_ONLY:
    print('[RUN_ROUND4_K_ONLY] Skipping exploratory data analysis (no sample DataFrame).')
else:
    print('=' * 70)
    print('EXPLORATORY DATA ANALYSIS \u2014 AliCCP / ESMM')
    print('=' * 70)

    # 1. Dataset validation
    print('\n--- 1. Dataset Validation ---')
    paper_stats = {'impressions': 84_000_000, 'clicks': 3_400_000, 'conversions': 18_000,
                   'users': 400_000, 'items': 4_300_000}
    total_rows = len(df_train) + len(df_test)
    total_clicks = df_train['click'].sum() + df_test['click'].sum()
    total_purchases = df_train['purchase'].sum() + df_test['purchase'].sum()
    print(f'Loaded rows:       {total_rows:>12,}  (paper: {paper_stats["impressions"]:,}  coverage: {total_rows/paper_stats["impressions"]*100:.2f}%)')
    print(f'Clicks:            {total_clicks:>12,}  (paper: {paper_stats["clicks"]:,}  coverage: {total_clicks/paper_stats["clicks"]*100:.2f}%)')
    print(f'Conversions:       {total_purchases:>12,}  (paper: {paper_stats["conversions"]:,}  coverage: {total_purchases/paper_stats["conversions"]*100:.2f}%)')

    # 2. Label distributions
    print('\n--- 2. Label Distributions ---')
    click_rate = total_clicks / total_rows
    conversion_rate = total_purchases / total_rows
    ctcvr = total_purchases / max(total_clicks, 1)
    print(f'Click rate (CTR):           {click_rate:.4f}  (paper: ~0.04)')
    print(f'Conversion rate (overall):  {conversion_rate:.6f}  (paper: ~0.0002)')
    print(f'Post-click CVR:             {ctcvr:.4f}  (paper: ~0.005)')
    for split_name, df_split in [('Train', df_train), ('Test', df_test)]:
        n = len(df_split)
        nc = df_split['click'].sum()
        np_ = df_split['purchase'].sum()
        print(f'  {split_name}: {n:,} rows, {nc:,} clicks ({nc/n:.4f}), {np_:,} conversions ({np_/n:.6f})')

    # 3. Feature cardinalities
    print('\n--- 3. Feature Cardinalities (sparse) ---')
    card_data = []
    for col in SPARSE_COLS:
        if col in df_train.columns:
            n_unique = df_train[col].nunique()
            card_data.append({'feature': col, 'cardinality': n_unique})
    card_df = pd.DataFrame(card_data)
    print(card_df.to_string(index=False))
    total_embed_params = card_df['cardinality'].sum() * EMBED_DIM
    print(f'\nTotal embedding parameters: {total_embed_params:,} (embed_dim={EMBED_DIM})')
    print(f'Embedding table memory: {total_embed_params * 4 / 1024**2:.1f} MB (float32)')

    # 4. Dense feature statistics
    print('\n--- 4. Dense Feature Statistics ---')
    dense_present = [c for c in DENSE_FEAT_COLS if c in df_train.columns]
    if dense_present:
        desc = df_train[dense_present].astype(float).describe().T
        print(desc[['count', 'mean', 'std', 'min', 'max']].to_string())
    else:
        print('No dense feature columns found in DataFrame.')

    # 5. Data sparsity
    print('\n--- 5. Data Sparsity ---')
    feature_cols = [c for c in SPARSE_COLS + DENSE_FEAT_COLS if c in df_train.columns]
    missing_rates = (df_train[feature_cols] == '0').mean()
    print(f'Average missing/zero rate across features: {missing_rates.mean():.4f}')
    print('Per-feature missing rates (top 10):')
    print(missing_rates.sort_values(ascending=False).head(10).to_string())

    # 6. Sequential label validation
    print('\n--- 6. Sequential Label Validation ---')
    violations_train = ((df_train['click'] == 0) & (df_train['purchase'] == 1)).sum()
    violations_test = ((df_test['click'] == 0) & (df_test['purchase'] == 1)).sum()
    print(f'Violations (purchase=1 but click=0): train={violations_train}, test={violations_test}')
    if violations_train + violations_test == 0:
        print('OK: Sequential dependence holds (purchase=1 => click=1)')
    else:
        print('WARNING: Sequential dependence violated!')

    # 7. Data size and RAM budget analysis
    print('\n--- 7. Data Size / RAM Budget Analysis ---')
    bytes_per_row = df_train.memory_usage(deep=True).sum() / len(df_train)
    embed_mem_mb = total_embed_params * 4 / 1024**2
    sample_sizes = [500_000, 1_000_000, 2_000_000, 5_000_000, 10_000_000, 84_000_000]
    runtimes = {'T4 (free)': 12_000, 'L4 (Pro)': 53_000, 'A100 (Pro+)': 83_000}
    print(f'{"Sample Size":>12} | {"DataFrame MB":>12} | {"Embed MB":>10} | {"Tensors MB":>10} | {"Peak MB":>10} | {"T4 12GB":>8} | {"L4 53GB":>8} | {"A100 83GB":>8}')
    print('-' * 105)
    for ss in sample_sizes:
        df_mb = ss * bytes_per_row / 1024**2
        n_features = len(feature_cols)
        tensor_mb = ss * n_features * 4 * 3 / 1024**2
        peak_mb = df_mb + embed_mem_mb + tensor_mb
        fits = {name: 'YES' if peak_mb < ram else 'NO' for name, ram in runtimes.items()}
        print(f'{ss:>12,} | {df_mb:>12.0f} | {embed_mem_mb:>10.1f} | {tensor_mb:>10.0f} | {peak_mb:>10.0f} | {fits["T4 (free)"]:>8} | {fits["L4 (Pro)"]:>8} | {fits["A100 (Pro+)"]:>8}')

    # 8. Positive count report
    print('\n--- 8. Positive Count Report ---')
    train_clicks = df_train['click'].sum()
    train_conversions = df_train['purchase'].sum()
    test_clicks = df_test['click'].sum()
    test_conversions = df_test['purchase'].sum()
    print(f'Train: {train_clicks:,} clicks, {train_conversions:,} conversions')
    print(f'Test:  {test_clicks:,} clicks, {test_conversions:,} conversions')
    if train_conversions < 5000:
        print('WARNING: < 5k train conversions \u2014 model may struggle to learn CVR signal!')
    else:
        print(f'OK: {train_conversions:,} conversions should be sufficient for training.')

    # 9. Sampling strategy recommendation
    print('\n--- 9. Sampling Strategy Recommendation ---')
    print(f'Current sample: {len(df_train):,} train + {len(df_test):,} test = {total_rows:,} total')
    if train_conversions < 5000:
        print('RECOMMENDATION: Use negative downsampling \u2014 keep all clicks+conversions,')
        print('drop non-click impressions to fit memory. This preserves the CVR signal.')
    elif total_rows < 2_000_000:
        print('RECOMMENDATION: Increase SAMPLE_SIZE to at least 2M for better coverage.')
    else:
        print('RECOMMENDATION: Current sample size appears adequate. Proceed with training.')

In [ ]:
# ===================== SHARED UTILITIES =====================
# Model classes, loss functions, eval helpers, training loops.
# This cell is updated by experiment-code-change subagent per round.

import torch
import torch.nn as nn
import numpy as np
import math
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss
import time
from concurrent.futures import ThreadPoolExecutor

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

import pandas as pd

# --------------- Label encoding ---------------

def build_sparse_vocabs(df, sparse_cols):
    """Build label-encoding vocabularies from a DataFrame. Index 0 = UNK/unseen."""
    vocabs = {}
    cardinalities = []
    for col in sparse_cols:
        unique_vals = df[col].astype(str).unique()
        vocab = {v: i + 1 for i, v in enumerate(unique_vals)}
        vocabs[col] = vocab
        cardinalities.append(len(vocab))
    return vocabs, cardinalities


def encode_and_tensorize(df, vocabs, sparse_cols, dense_feat_cols, label_col):
    """Encode sparse features via vocabs, cast dense to float, return tensors.
    Sparse indices use int32 on CPU to halve RAM; models call .long() for Embedding."""
    sparse_arrays = []
    for col in sparse_cols:
        encoded = df[col].astype(str).map(vocabs[col]).fillna(0).astype(np.int32).values
        sparse_arrays.append(encoded)
    sparse_t = torch.from_numpy(np.column_stack(sparse_arrays).astype(np.int32))
    dense_t = torch.from_numpy(
        df[dense_feat_cols].apply(pd.to_numeric, errors='coerce')
        .fillna(0.0).values.astype(np.float32)
    )
    label_t = torch.from_numpy(df[label_col].values.astype(np.float32))
    return sparse_t, dense_t, label_t


def _precompute_sparse_encode_tables(vocabs, sparse_cols):
    """Per-column (categories_tuple, lookup_int32) for vectorized sparse encoding."""
    tables = {}
    for col in sparse_cols:
        v = vocabs[col]
        cats = tuple(v.keys())
        lookup = np.array([v[c] for c in cats], dtype=np.int32)
        tables[col] = (cats, lookup)
    return tables


def encode_and_tensorize_fast(df, enc_tables, sparse_cols, dense_feat_cols, label_col):
    """Same outputs as encode_and_tensorize; faster categorical path + contiguous dense."""
    sparse_arrays = []
    for col in sparse_cols:
        cats, lookup = enc_tables[col]
        c = pd.Categorical(df[col].astype(str), categories=cats)
        codes = c.codes.astype(np.int64, copy=False)
        enc = np.where(codes >= 0, lookup[codes], 0).astype(np.int32)
        sparse_arrays.append(enc)
    sparse_t = torch.from_numpy(np.column_stack(sparse_arrays).astype(np.int32, copy=False))
    dense_arr = (
        df[dense_feat_cols].apply(pd.to_numeric, errors='coerce').fillna(0.0).values.astype(np.float32)
    )
    dense_t = torch.from_numpy(np.ascontiguousarray(dense_arr))
    label_t = torch.from_numpy(df[label_col].values.astype(np.float32))
    return sparse_t, dense_t, label_t

# --------------- BASEModel ---------------

class BASEModel(nn.Module):
    """Paper-exact BASE CVR tower: Embed(18) per field -> concat dense -> MLP 360->200->80->1."""

    def __init__(self, field_cardinalities, num_dense, embed_dim=18,
                 hidden_dims=(360, 200, 80)):
        super().__init__()
        self.embeddings = nn.ModuleList([
            nn.Embedding(card + 1, embed_dim) for card in field_cardinalities
        ])
        input_dim = len(field_cardinalities) * embed_dim + num_dense
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.mlp = nn.Sequential(*layers)

    def forward(self, sparse_x, dense_x):
        sparse_x = sparse_x.long()
        embs = [self.embeddings[i](sparse_x[:, i]) for i in range(sparse_x.size(1))]
        x = torch.cat(embs + [dense_x], dim=1)
        return self.mlp(x).squeeze(1)

# --------------- Training ---------------

def _train_base_lr_at_step(
    step, total_steps, base_lr, warmup_steps, lr_schedule,
    steps_per_epoch, lr_step_epochs, lr_step_gamma, cosine_min_lr_ratio,
):
    if warmup_steps > 0 and step < warmup_steps:
        return base_lr * float(step + 1) / float(warmup_steps)
    t = step - warmup_steps
    Tpost = max(1, total_steps - warmup_steps)
    progress = min(1.0, float(t) / float(Tpost))
    if lr_schedule == 'constant':
        return base_lr
    if lr_schedule == 'cosine':
        eta_min = base_lr * cosine_min_lr_ratio
        return eta_min + (base_lr - eta_min) * 0.5 * (1.0 + math.cos(math.pi * progress))
    if lr_schedule == 'step':
        if lr_step_epochs is None or lr_step_epochs <= 0:
            return base_lr
        period = max(1, int(lr_step_epochs) * steps_per_epoch)
        n_decays = t // period
        return base_lr * (float(lr_step_gamma) ** float(n_decays))
    raise ValueError(f'Unknown lr_schedule={lr_schedule!r} (use constant, cosine, step)')


def train_model(model, sparse_train, dense_train, y_train,
                epochs=10, batch_size=1024, lr=1e-3,
                weight_decay=0.0,
                lr_schedule='constant',
                warmup_steps=0,
                lr_step_epochs=3,
                lr_step_gamma=0.1,
                cosine_min_lr_ratio=0.01):
    """Train with BCEWithLogitsLoss + Adam. Returns per-epoch average losses.

    lr_schedule: 'constant' (default, legacy), 'cosine', or 'step' (decay every lr_step_epochs epochs).
    warmup_steps: linear warmup batches; 0 disables.
    """
    model.to(device)
    n_samples = len(sparse_train)
    steps_per_epoch = max(1, (n_samples + batch_size - 1) // batch_size)
    total_steps = epochs * steps_per_epoch

    optimizer = torch.optim.Adam(
        model.parameters(), lr=lr, betas=(0.9, 0.999), eps=1e-8, weight_decay=weight_decay,
    )
    criterion = nn.BCEWithLogitsLoss()
    loader = DataLoader(
        TensorDataset(sparse_train, dense_train, y_train),
        batch_size=batch_size, shuffle=True,
    )
    losses = []
    global_step = 0
    for epoch in range(epochs):
        model.train()
        total_loss, n_batches = 0.0, 0
        t_epoch = time.perf_counter()
        for sp, dn, y in loader:
            lr_now = _train_base_lr_at_step(
                global_step, total_steps, lr, warmup_steps, lr_schedule,
                steps_per_epoch, lr_step_epochs, lr_step_gamma, cosine_min_lr_ratio,
            )
            for pg in optimizer.param_groups:
                pg['lr'] = lr_now

            sp, dn, y = sp.to(device), dn.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(sp, dn)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            n_batches += 1
            global_step += 1
        avg = total_loss / n_batches
        losses.append(avg)
        dt = time.perf_counter() - t_epoch
        sps = n_samples / dt if dt > 0 else 0.0
        print(f'  Epoch {epoch+1}/{epochs}: loss={avg:.4f}  ({sps:,.0f} samples/s)')
    return losses

# --------------- Evaluation ---------------

def evaluate_auc(model, sparse_test, dense_test, y_test, batch_size=2048):
    """Compute ROC-AUC. Returns (auc, predictions_array)."""
    model.eval()
    loader = DataLoader(
        TensorDataset(sparse_test, dense_test, y_test),
        batch_size=batch_size, shuffle=False,
    )
    all_preds, all_labels = [], []
    with torch.no_grad():
        for sp, dn, y in loader:
            sp, dn, y = sp.to(device), dn.to(device), y.to(device)
            preds = torch.sigmoid(model(sp, dn))
            all_preds.append(preds.cpu().numpy())
            all_labels.append(y.cpu().numpy())
    preds_arr = np.concatenate(all_preds)
    labels_arr = np.concatenate(all_labels)
    return roc_auc_score(labels_arr, preds_arr), preds_arr

# --------------- ESMMModel ---------------

class ESMMModel(nn.Module):
    """ESMM two-tower (CTR+CVR) with shared embeddings. pCTCVR = pCTR * pCVR."""

    def __init__(self, field_cardinalities, num_dense, embed_dim=18,
                 hidden_dims=(360, 200, 80)):
        super().__init__()
        self.field_cardinalities = list(field_cardinalities)
        self.embed_dim = embed_dim
        self.num_fields = len(field_cardinalities)
        offsets = []
        off = 0
        for card in field_cardinalities:
            offsets.append(off)
            off += int(card) + 1
        self.register_buffer(
            'field_offsets',
            torch.tensor(offsets, dtype=torch.long).view(1, -1),
        )
        total_vocab = off
        self.unified_emb = nn.Embedding(total_vocab, embed_dim)
        input_dim = self.num_fields * embed_dim + num_dense

        ctr_layers = []
        prev = input_dim
        for h in hidden_dims:
            ctr_layers.append(nn.Linear(prev, h))
            ctr_layers.append(nn.ReLU())
            prev = h
        ctr_layers.append(nn.Linear(prev, 1))
        self.ctr_tower = nn.Sequential(*ctr_layers)

        cvr_layers = []
        prev = input_dim
        for h in hidden_dims:
            cvr_layers.append(nn.Linear(prev, h))
            cvr_layers.append(nn.ReLU())
            prev = h
        cvr_layers.append(nn.Linear(prev, 1))
        self.cvr_tower = nn.Sequential(*cvr_layers)

    def forward(self, sparse_x, dense_x):
        idx = sparse_x.long() + self.field_offsets
        e = self.unified_emb(idx)
        x = torch.cat([e.flatten(1), dense_x], dim=1)
        p_ctr = torch.sigmoid(self.ctr_tower(x).squeeze(1)).clamp(1e-7, 1 - 1e-7)
        p_cvr = torch.sigmoid(self.cvr_tower(x).squeeze(1)).clamp(1e-7, 1 - 1e-7)
        p_ctcvr = (p_ctr * p_cvr).clamp(1e-7, 1 - 1e-7)
        return p_ctr, p_cvr, p_ctcvr


# --------------- ESMM variants (shared bottom / MMoE / PLE) ---------------

def _init_linear(layer: nn.Linear) -> None:
    nn.init.kaiming_uniform_(layer.weight, a=5 ** 0.5)
    if layer.bias is not None:
        nn.init.zeros_(layer.bias)


class _ESMMExpertMLP(nn.Module):
    """Single hidden-layer expert: d_in -> hidden -> d_model (+ LayerNorm), PLE-style."""

    def __init__(self, d_in: int, hidden: int, d_model: int, dropout: float = 0.0) -> None:
        super().__init__()
        self.fc1 = nn.Linear(d_in, hidden)
        self.act = nn.ReLU()
        self.drop = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden, d_model)
        self.ln = nn.LayerNorm(d_model)
        _init_linear(self.fc1)
        _init_linear(self.fc2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        return self.ln(x)


class _ESMMGate(nn.Module):
    def __init__(self, d_selector: int, num_experts: int) -> None:
        super().__init__()
        self.linear = nn.Linear(d_selector, num_experts)
        _init_linear(self.linear)

    def forward(self, selector: torch.Tensor) -> torch.Tensor:
        return torch.softmax(self.linear(selector), dim=-1)


class _ESMM_PLELevel(nn.Module):
    """One PLE level: shared + task1 + task2 experts; three gates over all experts (ple_experiment/model.py)."""

    def __init__(
        self,
        d_in: int,
        d_model: int,
        expert_hidden: int,
        num_shared_experts: int,
        num_task_experts: int,
        dropout: float = 0.0,
        d_selector_t1=None,
        d_selector_t2=None,
        d_selector_shared=None,
    ) -> None:
        super().__init__()
        E_s = max(0, int(num_shared_experts))
        E_t = max(0, int(num_task_experts))
        self.shared_experts = nn.ModuleList(
            [_ESMMExpertMLP(d_in, expert_hidden, d_model, dropout) for _ in range(E_s)]
        )
        self.t1_experts = nn.ModuleList(
            [_ESMMExpertMLP(d_in, expert_hidden, d_model, dropout) for _ in range(E_t)]
        )
        self.t2_experts = nn.ModuleList(
            [_ESMMExpertMLP(d_in, expert_hidden, d_model, dropout) for _ in range(E_t)]
        )
        total_experts = E_s + E_t + E_t
        d_sel_t1 = d_selector_t1 if d_selector_t1 is not None else d_in
        d_sel_t2 = d_selector_t2 if d_selector_t2 is not None else d_in
        d_sel_sh = d_selector_shared if d_selector_shared is not None else d_in
        self.gate_t1 = _ESMMGate(d_sel_t1, total_experts)
        self.gate_t2 = _ESMMGate(d_sel_t2, total_experts)
        self.gate_shared = _ESMMGate(d_sel_sh, total_experts)

    def forward(self, x_expert, sel_t1, sel_t2, sel_shared):
        outs = []
        outs += [e(x_expert) for e in self.shared_experts]
        outs += [e(x_expert) for e in self.t1_experts]
        outs += [e(x_expert) for e in self.t2_experts]
        if len(outs) == 0:
            raise RuntimeError('PLELevel must have at least one expert')
        stacked = torch.stack(outs, dim=1)
        g_t1 = (self.gate_t1(sel_t1).unsqueeze(-1) * stacked).sum(dim=1)
        g_t2 = (self.gate_t2(sel_t2).unsqueeze(-1) * stacked).sum(dim=1)
        g_sh = (self.gate_shared(sel_shared).unsqueeze(-1) * stacked).sum(dim=1)
        return g_t1, g_t2, g_sh


class _ESMM_PLETower(nn.Module):
    def __init__(self, d_model: int, out_dim: int = 1) -> None:
        super().__init__()
        hidden = max(1, d_model // 2)
        self.fc1 = nn.Linear(d_model, hidden)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(hidden, out_dim)
        _init_linear(self.fc1)
        _init_linear(self.fc2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.act(self.fc1(x))
        return self.fc2(x)


class ESMM_SharedBottom(nn.Module):
    """ESMM with one shared trunk then separate CTR/CVR heads (same unified embeddings as ESMMModel)."""

    def __init__(
        self,
        field_cardinalities,
        num_dense,
        embed_dim=18,
        trunk_dims=(360, 200, 80),
    ):
        super().__init__()
        self.field_cardinalities = list(field_cardinalities)
        self.embed_dim = embed_dim
        self.num_fields = len(field_cardinalities)
        offsets, off = [], 0
        for card in field_cardinalities:
            offsets.append(off)
            off += int(card) + 1
        self.register_buffer('field_offsets', torch.tensor(offsets, dtype=torch.long).view(1, -1))
        self.unified_emb = nn.Embedding(off, embed_dim)
        input_dim = self.num_fields * embed_dim + num_dense
        trunk_layers = []
        prev = input_dim
        for h in trunk_dims:
            trunk_layers += [nn.Linear(prev, h), nn.ReLU()]
            prev = h
        self.shared_trunk = nn.Sequential(*trunk_layers)
        self.ctr_head = nn.Linear(prev, 1)
        self.cvr_head = nn.Linear(prev, 1)
        _init_linear(self.ctr_head)
        _init_linear(self.cvr_head)

    def forward(self, sparse_x, dense_x):
        idx = sparse_x.long() + self.field_offsets
        e = self.unified_emb(idx)
        x = torch.cat([e.flatten(1), dense_x], dim=1)
        h = self.shared_trunk(x)
        p_ctr = torch.sigmoid(self.ctr_head(h).squeeze(1)).clamp(1e-7, 1 - 1e-7)
        p_cvr = torch.sigmoid(self.cvr_head(h).squeeze(1)).clamp(1e-7, 1 - 1e-7)
        p_ctcvr = (p_ctr * p_cvr).clamp(1e-7, 1 - 1e-7)
        return p_ctr, p_cvr, p_ctcvr


class ESMM_MMoE(nn.Module):
    """Single-level MMoE: separate gates for CTR vs CVR over shared experts (same embedding front as ESMMModel)."""

    def __init__(
        self,
        field_cardinalities,
        num_dense,
        embed_dim=18,
        num_experts=4,
        expert_hidden=360,
        d_model=128,
        tower_hidden_ratio=0.5,
        dropout=0.0,
    ):
        super().__init__()
        self.field_cardinalities = list(field_cardinalities)
        self.embed_dim = embed_dim
        self.num_fields = len(field_cardinalities)
        offsets, off = [], 0
        for card in field_cardinalities:
            offsets.append(off)
            off += int(card) + 1
        self.register_buffer('field_offsets', torch.tensor(offsets, dtype=torch.long).view(1, -1))
        self.unified_emb = nn.Embedding(off, embed_dim)
        d_in = self.num_fields * embed_dim + num_dense
        E = int(num_experts)
        self.experts = nn.ModuleList(
            [_ESMMExpertMLP(d_in, expert_hidden, d_model, dropout) for _ in range(E)]
        )
        self.gate_ctr = _ESMMGate(d_in, E)
        self.gate_cvr = _ESMMGate(d_in, E)
        th = max(1, int(d_model * tower_hidden_ratio))
        self.ctr_tower = nn.Sequential(
            nn.Linear(d_model, th), nn.ReLU(), nn.Linear(th, 1),
        )
        self.cvr_tower = nn.Sequential(
            nn.Linear(d_model, th), nn.ReLU(), nn.Linear(th, 1),
        )
        for m in list(self.ctr_tower.modules()) + list(self.cvr_tower.modules()):
            if isinstance(m, nn.Linear):
                _init_linear(m)

    def forward(self, sparse_x, dense_x):
        idx = sparse_x.long() + self.field_offsets
        e = self.unified_emb(idx)
        x = torch.cat([e.flatten(1), dense_x], dim=1)
        expert_outs = torch.stack([ex(x) for ex in self.experts], dim=1)
        w_ctr = self.gate_ctr(x).unsqueeze(-1)
        w_cvr = self.gate_cvr(x).unsqueeze(-1)
        h_ctr = (w_ctr * expert_outs).sum(dim=1)
        h_cvr = (w_cvr * expert_outs).sum(dim=1)
        p_ctr = torch.sigmoid(self.ctr_tower(h_ctr).squeeze(1)).clamp(1e-7, 1 - 1e-7)
        p_cvr = torch.sigmoid(self.cvr_tower(h_cvr).squeeze(1)).clamp(1e-7, 1 - 1e-7)
        p_ctcvr = (p_ctr * p_cvr).clamp(1e-7, 1 - 1e-7)
        return p_ctr, p_cvr, p_ctcvr


class ESMM_PLE(nn.Module):
    """Two-level PLE for CTR (task1) and CVR (task2); selectors follow ple_experiment/model.py PLEModel."""

    def __init__(
        self,
        field_cardinalities,
        num_dense,
        embed_dim=18,
        d_model=128,
        expert_hidden=256,
        num_shared_experts=1,
        num_task_experts=1,
        dropout=0.0,
    ):
        super().__init__()
        self.field_cardinalities = list(field_cardinalities)
        self.embed_dim = embed_dim
        self.num_fields = len(field_cardinalities)
        offsets, off = [], 0
        for card in field_cardinalities:
            offsets.append(off)
            off += int(card) + 1
        self.register_buffer('field_offsets', torch.tensor(offsets, dtype=torch.long).view(1, -1))
        self.unified_emb = nn.Embedding(off, embed_dim)
        d_in = self.num_fields * embed_dim + num_dense
        ns, nt = int(num_shared_experts), int(num_task_experts)
        self.level1 = _ESMM_PLELevel(
            d_in, d_model, expert_hidden, ns, nt, dropout,
            d_selector_t1=d_in, d_selector_t2=d_in, d_selector_shared=d_in,
        )
        self.level2 = _ESMM_PLELevel(
            d_in, d_model, expert_hidden, ns, nt, dropout,
            d_selector_t1=d_model, d_selector_t2=d_model, d_selector_shared=d_model,
        )
        self.tower_ctr = _ESMM_PLETower(d_model, 1)
        self.tower_cvr = _ESMM_PLETower(d_model, 1)

    def forward(self, sparse_x, dense_x):
        idx = sparse_x.long() + self.field_offsets
        e = self.unified_emb(idx)
        x = torch.cat([e.flatten(1), dense_x], dim=1)
        g1_t1, g1_t2, g1_sh = self.level1(x, x, x, x)
        g2_t1, g2_t2, g2_sh = self.level2(x, g1_t1, g1_t2, g1_sh)
        g2_t1 = torch.nan_to_num(g2_t1, nan=0.0, posinf=1e4, neginf=-1e4)
        g2_t2 = torch.nan_to_num(g2_t2, nan=0.0, posinf=1e4, neginf=-1e4)
        p_ctr = torch.sigmoid(self.tower_ctr(g2_t1).squeeze(1)).clamp(1e-7, 1 - 1e-7)
        p_cvr = torch.sigmoid(self.tower_cvr(g2_t2).squeeze(1)).clamp(1e-7, 1 - 1e-7)
        p_ctcvr = (p_ctr * p_cvr).clamp(1e-7, 1 - 1e-7)
        return p_ctr, p_cvr, p_ctcvr


# --------------- ESMM Training ---------------

def train_esmm(model, sparse_train, dense_train, y_click, y_purchase,
               epochs=10, batch_size=1024, lr=1e-3):
    """Entire-space multi-task loss: BCE(click, pCTR) + BCE(click*purchase, pCTCVR)."""
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.999), eps=1e-8)
    y_ctcvr = y_click * y_purchase
    loader = DataLoader(
        TensorDataset(sparse_train, dense_train, y_click, y_ctcvr),
        batch_size=batch_size, shuffle=True,
    )
    losses = []
    n_samples_esmm = len(sparse_train)
    for epoch in range(epochs):
        model.train()
        total_loss, n_batches = 0.0, 0
        t_epoch = time.perf_counter()
        for sp, dn, yc, ycc in loader:
            sp, dn, yc, ycc = sp.to(device), dn.to(device), yc.to(device), ycc.to(device)
            optimizer.zero_grad()
            p_ctr, _, p_ctcvr = model(sp, dn)
            loss = (nn.functional.binary_cross_entropy(p_ctr, yc) +
                    nn.functional.binary_cross_entropy(p_ctcvr, ycc))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            n_batches += 1
        avg = total_loss / n_batches
        losses.append(avg)
        dt = time.perf_counter() - t_epoch
        sps = n_samples_esmm / dt if dt > 0 else 0.0
        print(f'  Epoch {epoch+1}/{epochs}: loss={avg:.4f}  ({sps:,.0f} samples/s)')
    return losses

# --------------- ESMM Evaluation ---------------

def evaluate_esmm_cvr(model, sparse_test, dense_test, y_purchase, batch_size=2048):
    """CVR AUC: pCVR vs purchase labels on clicked-only data."""
    model.eval()
    loader = DataLoader(
        TensorDataset(sparse_test, dense_test, y_purchase),
        batch_size=batch_size, shuffle=False,
    )
    all_preds, all_labels = [], []
    with torch.no_grad():
        for sp, dn, y in loader:
            sp, dn = sp.to(device), dn.to(device)
            _, p_cvr, _ = model(sp, dn)
            all_preds.append(p_cvr.cpu().numpy())
            all_labels.append(y.numpy())
    preds_arr = np.concatenate(all_preds)
    labels_arr = np.concatenate(all_labels)
    return roc_auc_score(labels_arr, preds_arr), preds_arr


def evaluate_esmm_ctcvr(model, sparse_test, dense_test, y_ctcvr, batch_size=2048):
    """CTCVR AUC: pCTCVR vs (click & purchase) labels on all data."""
    model.eval()
    loader = DataLoader(
        TensorDataset(sparse_test, dense_test, y_ctcvr),
        batch_size=batch_size, shuffle=False,
    )
    all_preds, all_labels = [], []
    with torch.no_grad():
        for sp, dn, y in loader:
            sp, dn = sp.to(device), dn.to(device)
            _, _, p_ctcvr = model(sp, dn)
            all_preds.append(p_ctcvr.cpu().numpy())
            all_labels.append(y.numpy())
    preds_arr = np.concatenate(all_preds)
    labels_arr = np.concatenate(all_labels)
    return roc_auc_score(labels_arr, preds_arr), preds_arr

# --------------- Focal Loss ---------------

class FocalLoss(nn.Module):
    """Focal loss for class-imbalanced binary classification (Lin et al., 2017)."""

    def __init__(self, gamma=2.0, alpha=0.25):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        p = torch.sigmoid(logits)
        pt = targets * p + (1 - targets) * (1 - p)
        alpha_t = targets * self.alpha + (1 - targets) * (1 - self.alpha)
        focal_weight = alpha_t * (1 - pt) ** self.gamma
        return (focal_weight * bce).mean()

# --------------- Training with Focal Loss ---------------

def train_model_focal(model, sparse_train, dense_train, y_train,
                      epochs=10, batch_size=1024, lr=1e-3, gamma=2.0, alpha=0.25):
    """Train with FocalLoss + Adam. Returns per-epoch average losses."""
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.999), eps=1e-8)
    criterion = FocalLoss(gamma=gamma, alpha=alpha)
    loader = DataLoader(
        TensorDataset(sparse_train, dense_train, y_train),
        batch_size=batch_size, shuffle=True,
    )
    losses = []
    n_samples_focal = len(sparse_train)
    for epoch in range(epochs):
        model.train()
        total_loss, n_batches = 0.0, 0
        t_epoch = time.perf_counter()
        for sp, dn, y in loader:
            sp, dn, y = sp.to(device), dn.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(sp, dn)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            n_batches += 1
        avg = total_loss / n_batches
        losses.append(avg)
        dt = time.perf_counter() - t_epoch
        sps = n_samples_focal / dt if dt > 0 else 0.0
        print(f'  Epoch {epoch+1}/{epochs}: loss={avg:.4f}  ({sps:,.0f} samples/s)')
    return losses


# --------------- ESMM eval streaming (full test — avoid giant pandas) ---------------

def evaluate_esmm_cvr_streaming_parquet(model, parquet_path, vocabs, sparse_cols, dense_feat_cols, batch_rows=None):
    import pyarrow.parquet as pq
    if batch_rows is None:
        batch_rows = EVAL_TEST_BATCH_ROWS
    cols = sparse_cols + dense_feat_cols + ['click', 'purchase']
    model.eval()
    enc_tables = _precompute_sparse_encode_tables(vocabs, sparse_cols)
    pf = pq.ParquetFile(parquet_path)
    all_p, all_y = [], []
    for batch in pf.iter_batches(batch_size=batch_rows, columns=cols):
        df = batch.to_pandas()
        del batch
        m = (df['click'].values == 1)
        if not m.any():
            del df
            continue
        df_c = df.loc[m].reset_index(drop=True)
        del df
        sp, dn, y = encode_and_tensorize_fast(df_c, enc_tables, sparse_cols, dense_feat_cols, 'purchase')
        del df_c
        loader = DataLoader(TensorDataset(sp, dn, y), batch_size=4096, shuffle=False)
        with torch.no_grad():
            for spb, dnb, yb in loader:
                spb, dnb = spb.to(device), dnb.to(device)
                _, pc, _ = model(spb, dnb)
                all_p.append(pc.cpu().numpy())
                all_y.append(yb.numpy())
        del sp, dn, y
    preds = np.concatenate(all_p)
    labels = np.concatenate(all_y)
    return roc_auc_score(labels, preds), preds


def evaluate_esmm_ctcvr_streaming_parquet(model, parquet_path, vocabs, sparse_cols, dense_feat_cols, batch_rows=None):
    import pyarrow.parquet as pq
    if batch_rows is None:
        batch_rows = EVAL_TEST_BATCH_ROWS
    cols = sparse_cols + dense_feat_cols + ['click', 'purchase']
    model.eval()
    enc_tables = _precompute_sparse_encode_tables(vocabs, sparse_cols)
    pf = pq.ParquetFile(parquet_path)
    all_p, all_y = [], []
    for batch in pf.iter_batches(batch_size=batch_rows, columns=cols):
        df = batch.to_pandas()
        del batch
        sp, dn, _ = encode_and_tensorize_fast(df, enc_tables, sparse_cols, dense_feat_cols, 'purchase')
        y_ct = torch.from_numpy(
            (df['click'].values * df['purchase'].values).astype(np.float32))
        del df
        loader = DataLoader(TensorDataset(sp, dn, y_ct), batch_size=4096, shuffle=False)
        with torch.no_grad():
            for spb, dnb, yb in loader:
                spb, dnb = spb.to(device), dnb.to(device)
                _, _, pct = model(spb, dnb)
                all_p.append(pct.cpu().numpy())
                all_y.append(yb.numpy())
        del sp, dn, y_ct

    preds = np.concatenate(all_p)
    labels = np.concatenate(all_y)
    return roc_auc_score(labels, preds), preds


def binary_pr_auc(labels, probs):
    '''Average precision (PR-AUC) for binary labels in {0,1}.'''
    y = np.asarray(labels).ravel()
    p = np.asarray(probs, dtype=np.float64).ravel()
    if len(np.unique(y)) < 2:
        return float('nan')
    return float(average_precision_score(y, p))


def binary_bce_log_loss(labels, probs):
    '''Sklearn log loss for binary probabilities (matches BCE on probabilities).'''
    y = np.asarray(labels).ravel()
    p = np.clip(np.asarray(probs, dtype=np.float64).ravel(), 1e-9, 1 - 1e-9)
    if len(np.unique(y)) < 2:
        return float('nan')
    return float(log_loss(y, p, labels=[0, 1]))


def expected_calibration_error(probs, labels, n_bins=15):
    '''ECE: mean |bin_confidence - bin_accuracy| weighted by bin mass.'''
    p = np.clip(np.asarray(probs, dtype=np.float64).ravel(), 1e-9, 1 - 1e-9)
    y = np.asarray(labels, dtype=np.float64).ravel()
    edges = np.linspace(0.0, 1.0, int(n_bins) + 1)
    ece = 0.0
    n = len(p)
    if n == 0:
        return float('nan')
    for i in range(int(n_bins)):
        lo, hi = edges[i], edges[i + 1]
        if i == int(n_bins) - 1:
            m = (p >= lo) & (p <= hi)
        else:
            m = (p >= lo) & (p < hi)
        cnt = int(m.sum())
        if cnt == 0:
            continue
        ece += (cnt / n) * abs(p[m].mean() - y[m].mean())
    return float(ece)


def evaluate_esmm_multitask_streaming_parquet(
    model, parquet_path, vocabs, sparse_cols, dense_feat_cols, batch_rows=None, ece_bins=15,
):
    '''One streaming pass over test Parquet: CTR / CTCVR / CVR (clicked-only) AUC, PR-AUC, log loss, ECE.'''
    import pyarrow.parquet as pq
    if batch_rows is None:
        batch_rows = EVAL_TEST_BATCH_ROWS
    cols = sparse_cols + dense_feat_cols + ['click', 'purchase']
    model.eval()
    enc_tables = _precompute_sparse_encode_tables(vocabs, sparse_cols)
    pf = pq.ParquetFile(parquet_path)
    ctr_p, ctr_y = [], []
    ctcvr_p, ctcvr_y = [], []
    cvr_p, cvr_y = [], []
    for batch in pf.iter_batches(batch_size=batch_rows, columns=cols):
        df = batch.to_pandas()
        del batch
        sp, dn, _ = encode_and_tensorize_fast(df, enc_tables, sparse_cols, dense_feat_cols, 'purchase')
        y_click = torch.from_numpy(df['click'].values.astype(np.float32))
        y_pur = torch.from_numpy(df['purchase'].values.astype(np.float32))
        y_ctcvr = y_click * y_pur
        del df
        loader = DataLoader(
            TensorDataset(sp, dn, y_click, y_pur, y_ctcvr),
            batch_size=4096, shuffle=False,
        )
        with torch.no_grad():
            for spb, dnb, ycb, ypb, yccb in loader:
                spb, dnb = spb.to(device), dnb.to(device)
                pc, pv, pcc = model(spb, dnb)
                pc_np = pc.cpu().numpy()
                pv_np = pv.cpu().numpy()
                pcc_np = pcc.cpu().numpy()
                yc_np = ycb.numpy()
                yp_np = ypb.numpy()
                ycc_np = yccb.numpy()
                ctr_p.append(pc_np)
                ctr_y.append(yc_np)
                ctcvr_p.append(pcc_np)
                ctcvr_y.append(ycc_np)
                m = yc_np > 0.5
                if m.any():
                    cvr_p.append(pv_np[m])
                    cvr_y.append(yp_np[m])
        del sp, dn, y_click, y_pur, y_ctcvr
    ctr_p = np.concatenate(ctr_p)
    ctr_y = np.concatenate(ctr_y)
    ctcvr_p = np.concatenate(ctcvr_p)
    ctcvr_y = np.concatenate(ctcvr_y)
    cvr_p = np.concatenate(cvr_p) if cvr_p else np.array([], dtype=np.float32)
    cvr_y = np.concatenate(cvr_y) if cvr_y else np.array([], dtype=np.float32)
    out = {}
    # CTR
    if len(np.unique(ctr_y)) >= 2:
        out['CTR_AUC'] = float(roc_auc_score(ctr_y, ctr_p))
        out['CTR_PR_AUC'] = binary_pr_auc(ctr_y, ctr_p)
        out['logloss_ctr'] = binary_bce_log_loss(ctr_y, ctr_p)
        out['ECE_ctr'] = expected_calibration_error(ctr_p, ctr_y, n_bins=ece_bins)
    else:
        out['CTR_AUC'] = float('nan')
        out['CTR_PR_AUC'] = float('nan')
        out['logloss_ctr'] = float('nan')
        out['ECE_ctr'] = float('nan')
    # CTCVR
    if len(np.unique(ctcvr_y)) >= 2:
        out['CTCVR_AUC'] = float(roc_auc_score(ctcvr_y, ctcvr_p))
        out['CTCVR_PR_AUC'] = binary_pr_auc(ctcvr_y, ctcvr_p)
        out['logloss_ctcvr'] = binary_bce_log_loss(ctcvr_y, ctcvr_p)
        out['ECE_ctcvr'] = expected_calibration_error(ctcvr_p, ctcvr_y, n_bins=ece_bins)
    else:
        out['CTCVR_AUC'] = float('nan')
        out['CTCVR_PR_AUC'] = float('nan')
        out['logloss_ctcvr'] = float('nan')
        out['ECE_ctcvr'] = float('nan')
    # CVR clicked-only
    if cvr_p.size > 0 and len(np.unique(cvr_y)) >= 2:
        out['CVR_AUC'] = float(roc_auc_score(cvr_y, cvr_p))
    else:
        out['CVR_AUC'] = float('nan')
    return out


# --------------- ESMM streaming training (Round 4 RAM) ---------------

# Consultants: avoid full 42M-row tensors; train from Parquet row groups + int32 host tensors.

R5_COMPILE_MODE = "default"  # string mode for torch.compile(..., mode=R5_COMPILE_MODE)


def encode_and_tensorize_arrow(table, enc_tables, sparse_cols, dense_feat_cols, label_col):
    """Same tensor dtypes/shapes as encode_and_tensorize_fast; input is pyarrow.Table."""
    import pyarrow as pa
    import pyarrow.compute as pc

    sparse_arrays = []
    for col in sparse_cols:
        cats, lookup = enc_tables[col]
        pa_col = table.column(col).combine_chunks()
        if pa.types.is_string(pa_col.type) or pa.types.is_large_string(pa_col.type):
            try:
                str_np = pa_col.to_numpy(zero_copy_only=False)
            except Exception:
                str_np = np.array(pa_col.to_pylist(), dtype=object)
        else:
            str_np = pc.cast(pa_col, pa.large_string()).to_numpy(zero_copy_only=False)
        s = pd.Series(str_np, dtype=object).astype(str)
        c = pd.Categorical(s, categories=list(cats))
        codes = c.codes.astype(np.int64, copy=False)
        enc = np.where(codes >= 0, lookup[codes], 0).astype(np.int32)
        sparse_arrays.append(enc)
    sparse_t = torch.from_numpy(np.column_stack(sparse_arrays).astype(np.int32, copy=False))

    dense_stack = []
    for cname in dense_feat_cols:
        x = table.column(cname).combine_chunks()
        if pa.types.is_null(x.type):
            arr = np.zeros(table.num_rows, dtype=np.float32)
        elif pa.types.is_floating(x.type) or pa.types.is_integer(x.type):
            arr = np.asarray(x.to_numpy(zero_copy_only=False), dtype=np.float64)
        else:
            arr = np.asarray(pc.cast(x, pa.float64()).to_numpy(zero_copy_only=False), dtype=np.float64)
        arr = np.nan_to_num(arr, nan=0.0).astype(np.float32)
        dense_stack.append(arr.reshape(-1, 1))
    dense_t = torch.from_numpy(np.ascontiguousarray(np.hstack(dense_stack)))

    ycol = table.column(label_col).combine_chunks()
    if pa.types.is_floating(ycol.type) or pa.types.is_integer(ycol.type):
        yarr = np.asarray(ycol.to_numpy(zero_copy_only=False), dtype=np.float32)
    else:
        yarr = np.asarray(pc.cast(ycol, pa.float32()).to_numpy(zero_copy_only=False), dtype=np.float32)
    label_t = torch.from_numpy(np.ascontiguousarray(yarr))
    return sparse_t, dense_t, label_t


def train_esmm_parquet_rowgroups(
    parquet_path, vocabs, field_cardinalities, sparse_cols, dense_feat_cols,
    epochs=5, batch_size=4096, lr=1e-3, seed=42,
    max_wall_seconds=None,
    max_optimizer_steps=None,
    max_batches_per_epoch=None,
    max_row_groups_per_epoch=None,
    use_amp=True,
    prefetch_row_groups=True,
    use_manual_batches=True,
    use_torch_compile=False,
    read_row_groups_as_arrow=False,
    model_ctor=None,
    model_ctor_kwargs=None,
):
    """One full pass over the file = one epoch; row-group order shuffled each epoch.

    Optional caps (None disables each): after each optimizer.step(), check limits and
    break with EARLY_STOP. Optimizer steps are cumulative across epochs; batch and
    row-group caps reset each epoch. Wall clock uses perf_counter from train start.

    use_amp: if True and CUDA, forward runs under autocast; BCE is computed in float32
    on detached head outputs to reduce underflow. Default True (no-op on CPU).

    prefetch_row_groups: if True, overlap Parquet decode/tensor prep for the next row
    group with training on the current (ThreadPoolExecutor max_workers=1, depth 1). Default True.

    use_manual_batches: if True, shuffle each row group with torch.randperm and slice
    batch_size chunks without DataLoader. If False, use DataLoader (legacy path). Default True.

    use_torch_compile: if True, CUDA, and torch>=2.0, wrap the model with torch.compile;
    on failure prints and keeps eager. Warmup steps run before the timed train span.

    model_ctor: optional callable (field_cardinalities, num_dense, embed_dim) -> nn.Module.
        Default builds ESMMModel(..., **model_ctor_kwargs).

    model_ctor_kwargs: optional dict of extra kwargs forwarded into model_ctor (or default ESMMModel).

    read_row_groups_as_arrow: if True, decode row groups with pyarrow only (no full
    DataFrame); falls back to pandas with a one-time message on first failure.
    """
    import random
    import pyarrow.parquet as pq

    _mkw = dict(model_ctor_kwargs or {})
    if model_ctor is None:
        model = ESMMModel(
            field_cardinalities, num_dense=len(dense_feat_cols), embed_dim=EMBED_DIM, **_mkw,
        )
    else:
        model = model_ctor(field_cardinalities, len(dense_feat_cols), EMBED_DIM, **_mkw)
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.999), eps=1e-8)
    use_amp_cuda = bool(use_amp and device.type == 'cuda')
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp_cuda)
    cols = sparse_cols + dense_feat_cols + ["click", "purchase"]
    pf = pq.ParquetFile(parquet_path)
    nrg = pf.num_row_groups
    if nrg == 0:
        raise ValueError(f"No row groups in {parquet_path}")
    n_train = int(pf.metadata.num_rows)
    enc_tables = _precompute_sparse_encode_tables(vocabs, sparse_cols)
    losses_out = []

    compiled_active = False
    if use_torch_compile:
        if torch.cuda.is_available():
            try:
                tv = torch.__version__.split('+')[0].split('.')
                major, minor = int(tv[0]), int(tv[1])
            except Exception:
                major, minor = 0, 0
            if (major, minor) >= (2, 0):
                try:
                    model = torch.compile(model, mode=R5_COMPILE_MODE)
                    compiled_active = True
                except Exception as e:
                    print(f'torch.compile failed ({e}); using eager ESMMModel.')
            else:
                print(f'torch.compile skipped: need torch>=2.0, got {torch.__version__}')
        else:
            print('torch.compile skipped: CUDA not available')

    arrow_fallback = [False]
    arrow_warned = {'printed': False}

    def _prepare_row_group_tensors(rg_idx):
        raw = pf.read_row_group(rg_idx, columns=cols)
        if read_row_groups_as_arrow and not arrow_fallback[0]:
            try:
                sp, dn, y_click = encode_and_tensorize_arrow(
                    raw, enc_tables, sparse_cols, dense_feat_cols, 'click')
                y_purchase = torch.from_numpy(
                    np.asarray(
                        raw.column('purchase').combine_chunks().to_numpy(zero_copy_only=False),
                        dtype=np.float32,
                    ))
                y_ctcvr = y_click * y_purchase
                del y_purchase
                return sp, dn, y_click, y_ctcvr
            except Exception as e:
                if not arrow_warned['printed']:
                    print(f'read_row_groups_as_arrow failed ({e}); falling back to pandas for remaining row groups.')
                    arrow_warned['printed'] = True
                arrow_fallback[0] = True
        sub = raw.to_pandas()
        sp, dn, y_click = encode_and_tensorize_fast(
            sub, enc_tables, sparse_cols, dense_feat_cols, 'click')
        y_purchase = torch.from_numpy(sub['purchase'].values.astype(np.float32))
        y_ctcvr = y_click * y_purchase
        del y_purchase
        del sub
        return sp, dn, y_click, y_ctcvr

    if compiled_active:
        try:
            sp0, dn0, yc0, ycc0 = _prepare_row_group_tensors(0)
            n0 = int(sp0.size(0))
            if n0 > 0:
                nw = min(int(batch_size), n0)
                for _ in range(3):
                    optimizer.zero_grad(set_to_none=True)
                    sp_b = sp0[:nw].to(device, non_blocking=True).long()
                    dn_b = dn0[:nw].to(device, non_blocking=True)
                    yc_b = yc0[:nw].to(device, non_blocking=True)
                    ycc_b = ycc0[:nw].to(device, non_blocking=True)
                    if use_amp_cuda:
                        with torch.amp.autocast('cuda', enabled=True):
                            p_ctr, _, p_ctcvr = model(sp_b, dn_b)
                        loss = (
                            nn.functional.binary_cross_entropy(p_ctr.float(), yc_b.float())
                            + nn.functional.binary_cross_entropy(p_ctcvr.float(), ycc_b.float())
                        )
                        scaler.scale(loss).backward()
                        scaler.step(optimizer)
                        scaler.update()
                    else:
                        p_ctr, _, p_ctcvr = model(sp_b, dn_b)
                        loss = (nn.functional.binary_cross_entropy(p_ctr, yc_b) +
                                nn.functional.binary_cross_entropy(p_ctcvr, ycc_b))
                        loss.backward()
                        optimizer.step()
            del sp0, dn0, yc0, ycc0
        except Exception as e:
            print(f'torch.compile warmup failed ({e}); continuing training.')

    early_reason = None
    opt_steps = 0
    samples_total_run = 0
    t_train_all = time.perf_counter()
    if device.type == 'cuda':
        torch.cuda.reset_peak_memory_stats()

    def _train_one_rg_tensors(sp, dn, y_click, y_ctcvr):
        nonlocal total_loss, n_batches, opt_steps, samples_this_epoch, batches_this_epoch
        nonlocal samples_total_run, early_reason

        def _step_batch(sp_b, dn_b, yc_b, ycc_b):
            nonlocal total_loss, n_batches, opt_steps, samples_this_epoch, batches_this_epoch
            nonlocal samples_total_run, early_reason
            optimizer.zero_grad(set_to_none=True)
            if use_amp_cuda:
                with torch.amp.autocast('cuda', enabled=True):
                    p_ctr, _, p_ctcvr = model(sp_b, dn_b)
                loss = (
                    nn.functional.binary_cross_entropy(p_ctr.float(), yc_b.float())
                    + nn.functional.binary_cross_entropy(p_ctcvr.float(), ycc_b.float())
                )
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                p_ctr, _, p_ctcvr = model(sp_b, dn_b)
                loss = (nn.functional.binary_cross_entropy(p_ctr, yc_b) +
                        nn.functional.binary_cross_entropy(p_ctcvr, ycc_b))
                loss.backward()
                optimizer.step()
            total_loss += loss.item()
            n_batches += 1
            opt_steps += 1
            bs = int(sp_b.size(0))
            batches_this_epoch += 1
            samples_this_epoch += bs
            samples_total_run += bs
            if max_wall_seconds is not None and (time.perf_counter() - t_train_all) >= max_wall_seconds:
                early_reason = 'max_wall_seconds'
                return True
            if max_optimizer_steps is not None and opt_steps >= max_optimizer_steps:
                early_reason = 'max_optimizer_steps'
                return True
            if max_batches_per_epoch is not None and batches_this_epoch >= max_batches_per_epoch:
                early_reason = 'max_batches_per_epoch'
                return True
            return False

        if use_manual_batches:
            n = int(sp.size(0))
            perm = torch.randperm(n)
            for start in range(0, n, batch_size):
                idx = perm[start:start + batch_size]
                sp_b = sp[idx].to(device, non_blocking=True).long()
                dn_b = dn[idx].to(device, non_blocking=True)
                yc_b = y_click[idx].to(device, non_blocking=True)
                ycc_b = y_ctcvr[idx].to(device, non_blocking=True)
                if _step_batch(sp_b, dn_b, yc_b, ycc_b):
                    return True
            return False

        loader = DataLoader(
            TensorDataset(sp, dn, y_click, y_ctcvr),
            batch_size=batch_size, shuffle=True, pin_memory=(device.type == 'cuda'),
        )
        for sp_b, dn_b, yc_b, ycc_b in loader:
            sp_b = sp_b.to(device, non_blocking=True).long()
            dn_b = dn_b.to(device, non_blocking=True)
            yc_b = yc_b.to(device, non_blocking=True)
            ycc_b = ycc_b.to(device, non_blocking=True)
            if _step_batch(sp_b, dn_b, yc_b, ycc_b):
                return True
        return False

    for epoch in range(epochs):
        if early_reason:
            break
        rng = list(range(nrg))
        random.seed(seed + epoch)
        random.shuffle(rng)
        model.train()
        total_loss, n_batches = 0.0, 0
        t_epoch = time.perf_counter()
        samples_this_epoch = 0
        batches_this_epoch = 0
        rgs_this_epoch = 0
        if prefetch_row_groups and len(rng) > 0:
            with ThreadPoolExecutor(max_workers=1) as _rg_ex:
                _fut = _rg_ex.submit(_prepare_row_group_tensors, rng[0])
                for rg_i in range(len(rng)):
                    rg = rng[rg_i]
                    if max_row_groups_per_epoch is not None and rgs_this_epoch >= max_row_groups_per_epoch:
                        early_reason = 'max_row_groups_per_epoch'
                        break
                    rgs_this_epoch += 1
                    sp, dn, y_click, y_ctcvr = _fut.result()
                    if rg_i + 1 < len(rng):
                        _fut = _rg_ex.submit(_prepare_row_group_tensors, rng[rg_i + 1])
                    rg_stop = _train_one_rg_tensors(sp, dn, y_click, y_ctcvr)
                    del sp, dn, y_click, y_ctcvr
                    if rg_stop:
                        break
        else:
            for rg in rng:
                if max_row_groups_per_epoch is not None and rgs_this_epoch >= max_row_groups_per_epoch:
                    early_reason = 'max_row_groups_per_epoch'
                    break
                rgs_this_epoch += 1
                sp, dn, y_click, y_ctcvr = _prepare_row_group_tensors(rg)
                rg_stop = _train_one_rg_tensors(sp, dn, y_click, y_ctcvr)
                del sp, dn, y_click, y_ctcvr
                if rg_stop:
                    break
        avg = total_loss / max(n_batches, 1)
        losses_out.append(avg)
        dt_ep = time.perf_counter() - t_epoch
        denom = samples_this_epoch if samples_this_epoch > 0 else n_train
        sps_ep = denom / dt_ep if dt_ep > 0 else 0.0
        print(f"  Epoch {epoch+1}/{epochs}: loss={avg:.4f} ({n_batches} batches)  ({sps_ep:,.0f} samples/s)")
        if early_reason:
            print(f'EARLY_STOP: reason={early_reason}')
            break
    dt_all = time.perf_counter() - t_train_all
    sps_all = samples_total_run / dt_all if dt_all > 0 else 0.0
    print(f'  Throughput (train span): {sps_all:,.0f} samples/s  ({samples_total_run:,} samples in {dt_all:.1f}s)')
    train_meta = {
        'early_stop_reason': early_reason,
        'samples_total_run': int(samples_total_run),
        'train_wall_seconds': float(dt_all),
        'samples_per_sec': float(sps_all),
        'use_amp': use_amp_cuda,
        'prefetch_row_groups': bool(prefetch_row_groups),
        'use_manual_batches': bool(use_manual_batches),
        'batch_size': int(batch_size),
        'use_torch_compile': bool(use_torch_compile),
        'torch_compile_active': bool(compiled_active),
        'read_row_groups_as_arrow': bool(read_row_groups_as_arrow),
        'read_row_groups_arrow_used': bool(read_row_groups_as_arrow and not arrow_fallback[0]),
    }
    if device.type == 'cuda':
        train_meta['cuda_max_memory_allocated_bytes'] = int(torch.cuda.max_memory_allocated())
    return model, losses_out, train_meta


def evaluate_esmm_cvr_indexed(model, sparse_all, dense_all, y_purchase_all, click_mask, batch_size=4096):
    """CVR AUC on clicked rows using a boolean mask over precomputed test tensors."""
    model.eval()
    sp_s = sparse_all[click_mask]
    dn_s = dense_all[click_mask]
    y_s = y_purchase_all[click_mask]
    loader = DataLoader(
        TensorDataset(sp_s, dn_s, y_s),
        batch_size=batch_size, shuffle=False,
    )
    all_preds, all_labels = [], []
    with torch.no_grad():
        for sp, dn, y in loader:
            sp, dn = sp.to(device), dn.to(device)
            _, p_cvr, _ = model(sp, dn)
            all_preds.append(p_cvr.cpu().numpy())
            all_labels.append(y.numpy())
    preds_arr = np.concatenate(all_preds)
    labels_arr = np.concatenate(all_labels)
    return roc_auc_score(labels_arr, preds_arr), preds_arr


# --------------- Frequency-filtered Vocabs (Round 4+) ---------------

def build_sparse_vocabs_filtered(df, sparse_cols, min_count=5):
    """Build label-encoding vocabularies with frequency filtering.
    IDs appearing fewer than min_count times map to index 0 (UNK)."""
    vocabs = {}
    cardinalities = []
    for col in sparse_cols:
        vals = df[col].astype(str)
        counts = vals.value_counts()
        total_unique = len(counts)
        kept = counts[counts >= min_count]
        vocab = {v: i + 1 for i, v in enumerate(kept.index)}
        vocabs[col] = vocab
        cardinalities.append(len(vocab))
        filtered = total_unique - len(kept)
        print(f'  {col}: {total_unique} unique, {len(kept)} kept (>={min_count}), {filtered} filtered')
    return vocabs, cardinalities

# --------------- Dense Feature Normalization (Round 4+) ---------------

def normalize_dense_features(df, dense_feat_cols):
    """Apply log1p normalization to dense features. Returns a copy with only
    dense_feat_cols transformed; all other columns are preserved as-is."""
    df_out = df.copy()
    for i, col in enumerate(dense_feat_cols):
        x = pd.to_numeric(df_out[col], errors='coerce').fillna(0.0)
        if i < 3:
            print(f'  {col} BEFORE: min={x.min():.4f}, max={x.max():.4f}, mean={x.mean():.4f}')
        x_norm = np.log1p(np.abs(x)) * np.sign(x)
        if i < 3:
            print(f'  {col} AFTER:  min={x_norm.min():.4f}, max={x_norm.max():.4f}, mean={x_norm.mean():.4f}')
        df_out[col] = x_norm
    return df_out


In [ ]:
import json, os, time

if RUN_ROUND4_K_ONLY:
    print('[RUN_ROUND4_K_ONLY] Skipping Round 1 (Experiment A).')
else:
    _ROUND_1_CACHE = os.path.join(ROUND_RESULTS_DIR, 'round_1_results.json')

    if not os.path.exists(_ROUND_1_CACHE):
        # ===================== ROUND 1 =====================
        results = {}

        # --- Data preparation ---
        print('Preparing data: building vocabs from all training data...')
        vocabs, cardinalities = build_sparse_vocabs(df_train, SPARSE_COLS)
        print(f'  Sparse fields: {len(SPARSE_COLS)}, Dense fields: {len(DENSE_FEAT_COLS)}')
        print(f'  Cardinalities: {cardinalities}')

        df_train_clicked = df_train[df_train['click'] == 1].reset_index(drop=True)
        df_test_clicked = df_test[df_test['click'] == 1].reset_index(drop=True)
        print(f'  Clicked-only: train={len(df_train_clicked):,}, test={len(df_test_clicked):,}')
        print(f'  Train CVR (purchase|click): {df_train_clicked["purchase"].mean():.4f}')
        print(f'  Test  CVR (purchase|click): {df_test_clicked["purchase"].mean():.4f}')

        sparse_train_t, dense_train_t, y_train_t = encode_and_tensorize(
            df_train_clicked, vocabs, SPARSE_COLS, DENSE_FEAT_COLS, 'purchase')
        sparse_test_t, dense_test_t, y_test_t = encode_and_tensorize(
            df_test_clicked, vocabs, SPARSE_COLS, DENSE_FEAT_COLS, 'purchase')

        # --- Experiment A: Faithful BASE model ---
        print('\n' + '=' * 60)
        print('EXPERIMENT A: BASE CVR Model (paper architecture)')
        print('=' * 60)
        t0 = time.time()

        model_a = BASEModel(cardinalities, num_dense=len(DENSE_FEAT_COLS), embed_dim=EMBED_DIM)
        input_dim = len(SPARSE_COLS) * EMBED_DIM + len(DENSE_FEAT_COLS)
        print(f'  Input dim: {input_dim} ({len(SPARSE_COLS)}x{EMBED_DIM} + {len(DENSE_FEAT_COLS)})')
        total_params = sum(p.numel() for p in model_a.parameters())
        print(f'  Total parameters: {total_params:,}')

        _ = train_model(model_a, sparse_train_t, dense_train_t, y_train_t,
                        epochs=10, batch_size=1024, lr=1e-3)

        cvr_auc, _ = evaluate_auc(model_a, sparse_test_t, dense_test_t, y_test_t)

        ctcvr_auc = None
        try:
            sparse_test_all, dense_test_all, y_test_all = encode_and_tensorize(
                df_test, vocabs, SPARSE_COLS, DENSE_FEAT_COLS, 'purchase')
            _, preds_all = evaluate_auc(model_a, sparse_test_all, dense_test_all, y_test_all)
            click_rate = df_train['click'].mean()
            pctcvr = click_rate * preds_all
            from sklearn.metrics import roc_auc_score as _auc
            ctcvr_auc = float(_auc(df_test['purchase'].values, pctcvr))
            print(f'  CTCVR_AUC (all test, pCTR~={click_rate:.4f}): {ctcvr_auc:.4f}')
        except Exception as e:
            print(f'  CTCVR_AUC computation skipped: {e}')

        elapsed = time.time() - t0
        results['A'] = {
            'CVR_AUC': float(cvr_auc),
            'CTCVR_AUC': ctcvr_auc,
            'wall_clock_seconds': int(elapsed),
        }
        print(f'\n  >> CVR_AUC={cvr_auc:.4f}, time={elapsed:.0f}s')

        with open(_ROUND_1_CACHE, 'w') as f:
            json.dump(results, f)

        print('\n' + '=' * 60)
        print('ROUND 1 SUMMARY')
        print('=' * 60)
        for name, m in results.items():
            extra = f", CTCVR_AUC={m['CTCVR_AUC']:.4f}" if m.get('CTCVR_AUC') else ''
            print(f"  {name}: CVR_AUC={m['CVR_AUC']:.4f}{extra} ({m['wall_clock_seconds']}s)")
    else:
        with open(_ROUND_1_CACHE) as f:
            results = json.load(f)
        print('ROUND 1: SKIPPED (cached)')
        for name, m in results.items():
            extra = f", CTCVR_AUC={m['CTCVR_AUC']:.4f}" if m.get('CTCVR_AUC') else ''
            print(f"  {name}: CVR_AUC={m['CVR_AUC']:.4f}{extra}")

In [ ]:
import json, os, time

if RUN_ROUND4_K_ONLY:
    print('[RUN_ROUND4_K_ONLY] Skipping Round 2 (Experiment D).')
else:
    _ROUND_2_CACHE = os.path.join(ROUND_RESULTS_DIR, 'round_2_results.json')

    if not os.path.exists(_ROUND_2_CACHE):
        # ===================== ROUND 2 =====================
        results = {}

        vocabs, cardinalities = build_sparse_vocabs(df_train, SPARSE_COLS)

        # Entire-space training data (all impressions)
        sparse_train_all, dense_train_all, y_click_train = encode_and_tensorize(
            df_train, vocabs, SPARSE_COLS, DENSE_FEAT_COLS, 'click')
        _, _, y_purchase_train = encode_and_tensorize(
            df_train, vocabs, SPARSE_COLS, DENSE_FEAT_COLS, 'purchase')

        # CVR evaluation: clicked-only test data
        df_test_clicked = df_test[df_test['click'] == 1].reset_index(drop=True)
        sparse_test_clicked, dense_test_clicked, y_purchase_test_clicked = encode_and_tensorize(
            df_test_clicked, vocabs, SPARSE_COLS, DENSE_FEAT_COLS, 'purchase')

        # CTCVR evaluation: all test data
        sparse_test_all, dense_test_all, _ = encode_and_tensorize(
            df_test, vocabs, SPARSE_COLS, DENSE_FEAT_COLS, 'purchase')
        y_ctcvr_test = torch.FloatTensor(
            (df_test['click'].values * df_test['purchase'].values).astype(np.float32))

        # --- Experiment D: ESMM (shared embeddings, multi-task loss) ---
        print('=' * 60)
        print('EXPERIMENT D: ESMM (shared embeddings, entire-space multi-task)')
        print('=' * 60)
        t0 = time.time()

        model_d = ESMMModel(cardinalities, num_dense=len(DENSE_FEAT_COLS), embed_dim=EMBED_DIM)
        total_params = sum(p.numel() for p in model_d.parameters())
        print(f'  Total parameters: {total_params:,}')

        _ = train_esmm(model_d, sparse_train_all, dense_train_all,
                       y_click_train, y_purchase_train,
                       epochs=10, batch_size=1024, lr=1e-3)

        cvr_auc_d, _ = evaluate_esmm_cvr(model_d, sparse_test_clicked, dense_test_clicked, y_purchase_test_clicked)
        print(f'  CVR_AUC (clicked-only test): {cvr_auc_d:.4f}')

        ctcvr_auc_d, _ = evaluate_esmm_ctcvr(model_d, sparse_test_all, dense_test_all, y_ctcvr_test)
        print(f'  CTCVR_AUC (all test): {ctcvr_auc_d:.4f}')

        elapsed_d = time.time() - t0
        results['D'] = {
            'CVR_AUC': float(cvr_auc_d),
            'CTCVR_AUC': float(ctcvr_auc_d),
            'wall_clock_seconds': int(elapsed_d)
        }
        print(f'  >> CVR_AUC={cvr_auc_d:.4f}, CTCVR_AUC={ctcvr_auc_d:.4f}, time={elapsed_d:.0f}s')

        with open(_ROUND_2_CACHE, 'w') as f:
            json.dump(results, f)

        print('\n' + '=' * 60)
        print(f'ROUND 2 SUMMARY (vs best so far: A=0.5622)')
        print('=' * 60)
        for name, m in results.items():
            print(f"  {name}: CVR_AUC={m['CVR_AUC']:.4f}, CTCVR_AUC={m['CTCVR_AUC']:.4f} ({m['wall_clock_seconds']}s)")
    else:
        with open(_ROUND_2_CACHE) as f:
            results = json.load(f)
        print('ROUND 2: SKIPPED (cached)')
        for name, m in results.items():
            print(f"  {name}: CVR_AUC={m['CVR_AUC']:.4f}, CTCVR_AUC={m.get('CTCVR_AUC', 'N/A')}")

In [ ]:
import json, os, time

if RUN_ROUND4_K_ONLY:
    print('[RUN_ROUND4_K_ONLY] Skipping Round 3 (Experiments G and H).')
else:
    _ROUND_3_CACHE = os.path.join(ROUND_RESULTS_DIR, 'round_3_results.json')

    if not os.path.exists(_ROUND_3_CACHE):
        # ===================== ROUND 3 =====================
        results = {}

        # --- Experiment G: BASE model with 15M data ---
        print('=' * 60)
        print('EXPERIMENT G: BASE CVR Model with 15M impressions')
        print('=' * 60)
        t0 = time.time()

        print('Loading 15M rows (this may take 10-15 min)...')
        df_train_15m, df_test_15m = load_or_parse_ali_ccp(DATA_DIR, 15_000_000, PROCESSED_PARSED_DIR)
        if df_train_15m is None or len(df_train_15m) == 0:
            raise RuntimeError('15M AliCCP load/parse failed; check raw files under DATA_DIR.')
        df_train_15m['click'] = df_train_15m['click'].astype(int)
        df_train_15m['purchase'] = df_train_15m['purchase'].astype(int)
        df_test_15m['click'] = df_test_15m['click'].astype(int)
        df_test_15m['purchase'] = df_test_15m['purchase'].astype(int)
        print(f'  15M data: train={len(df_train_15m):,}, test={len(df_test_15m):,}')
        print(f'  Train clicks: {df_train_15m["click"].sum():,}')
        print(f'  Train conversions: {df_train_15m["purchase"].sum():,}')

        vocabs_g, cards_g = build_sparse_vocabs(df_train_15m, SPARSE_COLS)
        df_train_clicked_g = df_train_15m[df_train_15m['click'] == 1].reset_index(drop=True)
        df_test_clicked_g = df_test_15m[df_test_15m['click'] == 1].reset_index(drop=True)
        print(f'  Clicked-only: train={len(df_train_clicked_g):,}, test={len(df_test_clicked_g):,}')
        print(f'  Train CVR: {df_train_clicked_g["purchase"].mean():.4f}')

        sp_tr_g, dn_tr_g, y_tr_g = encode_and_tensorize(
            df_train_clicked_g, vocabs_g, SPARSE_COLS, DENSE_FEAT_COLS, 'purchase')
        sp_te_g, dn_te_g, y_te_g = encode_and_tensorize(
            df_test_clicked_g, vocabs_g, SPARSE_COLS, DENSE_FEAT_COLS, 'purchase')

        model_g = BASEModel(cards_g, num_dense=len(DENSE_FEAT_COLS), embed_dim=EMBED_DIM)
        _ = train_model(model_g, sp_tr_g, dn_tr_g, y_tr_g, epochs=10, batch_size=1024, lr=1e-3)
        cvr_auc_g, _ = evaluate_auc(model_g, sp_te_g, dn_te_g, y_te_g)
        elapsed_g = time.time() - t0

        results['G'] = {
            'CVR_AUC': float(cvr_auc_g),
            'CTCVR_AUC': None,
            'wall_clock_seconds': int(elapsed_g)
        }
        print(f'\n  >> G: CVR_AUC={cvr_auc_g:.4f}, time={elapsed_g:.0f}s')

        del df_train_15m, df_test_15m, df_train_clicked_g, df_test_clicked_g
        del sp_tr_g, dn_tr_g, y_tr_g, sp_te_g, dn_te_g, y_te_g, model_g
        import gc; gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        # --- Experiment H: BASE model with focal loss on 5M data ---
        print('\n' + '=' * 60)
        print('EXPERIMENT H: BASE CVR Model with Focal Loss (5M data)')
        print('=' * 60)
        t0 = time.time()

        vocabs_h, cards_h = build_sparse_vocabs(df_train, SPARSE_COLS)
        df_train_clicked_h = df_train[df_train['click'] == 1].reset_index(drop=True)
        df_test_clicked_h = df_test[df_test['click'] == 1].reset_index(drop=True)

        sp_tr_h, dn_tr_h, y_tr_h = encode_and_tensorize(
            df_train_clicked_h, vocabs_h, SPARSE_COLS, DENSE_FEAT_COLS, 'purchase')
        sp_te_h, dn_te_h, y_te_h = encode_and_tensorize(
            df_test_clicked_h, vocabs_h, SPARSE_COLS, DENSE_FEAT_COLS, 'purchase')

        model_h = BASEModel(cards_h, num_dense=len(DENSE_FEAT_COLS), embed_dim=EMBED_DIM)
        _ = train_model_focal(model_h, sp_tr_h, dn_tr_h, y_tr_h,
                              epochs=10, batch_size=1024, lr=1e-3, gamma=2.0, alpha=0.25)
        cvr_auc_h, _ = evaluate_auc(model_h, sp_te_h, dn_te_h, y_te_h)
        elapsed_h = time.time() - t0

        results['H'] = {
            'CVR_AUC': float(cvr_auc_h),
            'CTCVR_AUC': None,
            'wall_clock_seconds': int(elapsed_h)
        }
        print(f'\n  >> H: CVR_AUC={cvr_auc_h:.4f}, time={elapsed_h:.0f}s')

        with open(_ROUND_3_CACHE, 'w') as f:
            json.dump(results, f)

        print('\n' + '=' * 60)
        print(f'ROUND 3 SUMMARY (vs best so far: A=0.5622)')
        print('=' * 60)
        for name, m in results.items():
            ctcvr_str = f", CTCVR_AUC={m['CTCVR_AUC']:.4f}" if m.get('CTCVR_AUC') else ''
            print(f"  {name}: CVR_AUC={m['CVR_AUC']:.4f}{ctcvr_str} ({m['wall_clock_seconds']}s)")
    else:
        with open(_ROUND_3_CACHE) as f:
            results = json.load(f)
        print('ROUND 3: SKIPPED (cached)')
        for name, m in results.items():
            ctcvr_str = f", CTCVR_AUC={m['CTCVR_AUC']:.4f}" if m.get('CTCVR_AUC') else ''
            print(f"  {name}: CVR_AUC={m['CVR_AUC']:.4f}{ctcvr_str}")

In [ ]:
import json, os, time, gc
import numpy as np
import torch
import pandas as pd

_ROUND_4_CACHE = os.path.join(
    ROUND_RESULTS_DIR,
    'round_4_k_only_results.json' if RUN_ROUND4_K_ONLY else 'round_4_results.json',
)
# R4_NORM_* paths come from Config cell.

# Hypothesis L — BASE (J) path only: LR / WD / warmup (defaults match legacy J).
R4_BASE_LR_MODE = 'constant'  # 'constant' | 'cosine' | 'step'
R4_BASE_WD = 0.0
R4_BASE_WARMUP_STEPS = 0
R4_BASE_COSINE_MIN_LR_RATIO = 0.01
R4_BASE_STEP_EPOCHS = 3
R4_BASE_STEP_GAMMA = 0.1

if not os.path.exists(_ROUND_4_CACHE):
    results = {}

    t0 = time.time()
    # Path-only full split: streaming parse if needed — never pd.read_parquet(full).
    p_train, p_test = ensure_full_split_parquet_streaming(DATA_DIR, PROCESSED_FULL_DIR)
    summarize_split(p_train, p_test)
    print(f'  Full split paths ready in {time.time()-t0:.1f}s')

    t1 = time.time()
    print('Frequency-filtered vocabs: load cache or Parquet scans (see R4_FILTERED_VOCAB_CACHE)...')
    vocabs_r4, cards_r4 = load_or_build_sparse_vocabs_filtered_parquet(
        p_train, SPARSE_COLS, min_count=5, cache_path=R4_FILTERED_VOCAB_CACHE,
        force_rebuild=FORCE_REBUILD_R4_VOCAB)
    print(f'  Vocab ready in {time.time()-t1:.1f}s')

    if not (os.path.isfile(R4_NORM_TRAIN) and os.path.isfile(R4_NORM_TEST)):
        print('Streaming dense log1p -> normalized Parquet...')
        t2 = time.time()
        stream_normalize_parquet(p_train, R4_NORM_TRAIN, SPARSE_COLS, DENSE_FEAT_COLS)
        stream_normalize_parquet(p_test, R4_NORM_TEST, SPARSE_COLS, DENSE_FEAT_COLS)
        print(f'  Normalize done in {time.time()-t2:.1f}s')
    else:
        print(f'Reusing on-disk {R4_NORM_TRAIN} / {R4_NORM_TEST} (set CLEAN_R4_NORMALIZED_PARQUET=True in Config to rebuild)')

    import psutil
    gc.collect()
    print(f'  RAM after prep: {psutil.Process().memory_info().rss / 1024**3:.1f} GB')

    if not RUN_ROUND4_K_ONLY:
            print('\n' + '=' * 60)
            print('EXPERIMENT J: BASE CVR (full split, freq filter, batch=4096, log1p, int32 sparse)')
            print(f'  BASE train (L toggles): mode={R4_BASE_LR_MODE!r}, wd={R4_BASE_WD}, warmup_steps={R4_BASE_WARMUP_STEPS}')
            print('=' * 60)
            t0 = time.time()

            df_tr_j = pd.read_parquet(R4_NORM_TRAIN, filters=[('click', '==', 1)])
            df_te_j = pd.read_parquet(R4_NORM_TEST, filters=[('click', '==', 1)])
            print(f'  Clicked-only: train={len(df_tr_j):,}, test={len(df_te_j):,}')
            print(f'  Train CVR: {df_tr_j["purchase"].mean():.4f}')

            sp_tr_j, dn_tr_j, y_tr_j = encode_and_tensorize(
                df_tr_j, vocabs_r4, SPARSE_COLS, DENSE_FEAT_COLS, 'purchase')
            del df_tr_j
            sp_te_j, dn_te_j, y_te_j = encode_and_tensorize(
                df_te_j, vocabs_r4, SPARSE_COLS, DENSE_FEAT_COLS, 'purchase')
            del df_te_j
            gc.collect()
            print(f'  RAM after J tensorize: {psutil.Process().memory_info().rss / 1024**3:.1f} GB')

            model_j = BASEModel(cards_r4, num_dense=len(DENSE_FEAT_COLS), embed_dim=EMBED_DIM)
            print(f'  Model params: {sum(p.numel() for p in model_j.parameters()):,}')
            _ = train_model(
                model_j, sp_tr_j, dn_tr_j, y_tr_j,
                epochs=10, batch_size=4096, lr=1e-3,
                weight_decay=R4_BASE_WD,
                lr_schedule=R4_BASE_LR_MODE,
                warmup_steps=R4_BASE_WARMUP_STEPS,
                lr_step_epochs=R4_BASE_STEP_EPOCHS,
                lr_step_gamma=R4_BASE_STEP_GAMMA,
                cosine_min_lr_ratio=R4_BASE_COSINE_MIN_LR_RATIO,
            )
            cvr_auc_j, _ = evaluate_auc(model_j, sp_te_j, dn_te_j, y_te_j)
            elapsed_j = time.time() - t0

            results['J'] = {
                'CVR_AUC': float(cvr_auc_j),
                'CTCVR_AUC': None,
                'wall_clock_seconds': int(elapsed_j)
            }
            print(f'\n  >> J: CVR_AUC={cvr_auc_j:.4f}, time={elapsed_j:.0f}s')

            del sp_tr_j, dn_tr_j, y_tr_j, sp_te_j, dn_te_j, y_te_j, model_j
            gc.collect()

    else:
        print('[RUN_ROUND4_K_ONLY] Skipping Experiment J (BASE); running Experiment K only.')

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print('\n' + '=' * 60)
    print('EXPERIMENT K: ESMM row-group streaming (5 epochs, batch=4096)')
    print('=' * 60)
    t0 = time.time()

    model_k, _, _k_train_meta = train_esmm_parquet_rowgroups(
        R4_NORM_TRAIN, vocabs_r4, cards_r4, SPARSE_COLS, DENSE_FEAT_COLS,
        epochs=5, batch_size=4096, lr=1e-3, seed=RANDOM_STATE,
        max_wall_seconds=K_EARLY_STOP_MAX_WALL_SECONDS,
        max_optimizer_steps=K_EARLY_STOP_MAX_OPTIMIZER_STEPS,
        max_batches_per_epoch=K_EARLY_STOP_MAX_BATCHES_PER_EPOCH,
        max_row_groups_per_epoch=K_EARLY_STOP_MAX_ROW_GROUPS_PER_EPOCH,
    )
    print(f"  K train throughput: {_k_train_meta['samples_per_sec']:,.0f} samples/s, "
          f"train_wall={_k_train_meta['train_wall_seconds']:.1f}s, "
          f"early_stop={_k_train_meta['early_stop_reason']!r}")

    cvr_auc_k, _ = evaluate_esmm_cvr_streaming_parquet(
        model_k, R4_NORM_TEST, vocabs_r4, SPARSE_COLS, DENSE_FEAT_COLS)
    print(f'  CVR_AUC (clicked-only test, streaming): {cvr_auc_k:.4f}')

    ctcvr_auc_k, _ = evaluate_esmm_ctcvr_streaming_parquet(
        model_k, R4_NORM_TEST, vocabs_r4, SPARSE_COLS, DENSE_FEAT_COLS)
    print(f'  CTCVR_AUC (all test, streaming): {ctcvr_auc_k:.4f}')

    elapsed_k = time.time() - t0
    results['K'] = {
        'CVR_AUC': float(cvr_auc_k),
        'CTCVR_AUC': float(ctcvr_auc_k),
        'wall_clock_seconds': int(elapsed_k)
    }
    print(f'\n  >> K: CVR_AUC={cvr_auc_k:.4f}, CTCVR_AUC={ctcvr_auc_k:.4f}, time={elapsed_k:.0f}s')

    del model_k
    gc.collect()

    with open(_ROUND_4_CACHE, 'w') as f:
        json.dump(results, f)

    print('\n' + '=' * 60)
    print('ROUND 4 SUMMARY (vs best so far: G=0.5841)')
    print('=' * 60)
    for name, m in results.items():
        ctcvr_str = f", CTCVR_AUC={m['CTCVR_AUC']:.4f}" if m.get('CTCVR_AUC') is not None else ''
        print(f"  {name}: CVR_AUC={m['CVR_AUC']:.4f}{ctcvr_str} ({m['wall_clock_seconds']}s)")
else:
    with open(_ROUND_4_CACHE) as f:
        results = json.load(f)
    print('ROUND 4: SKIPPED (cached)')
    for name, m in results.items():
        ctcvr_str = f", CTCVR_AUC={m['CTCVR_AUC']:.4f}" if m.get('CTCVR_AUC') is not None else ''
        print(f"  {name}: CVR_AUC={m['CVR_AUC']:.4f}{ctcvr_str}")


In [ ]:
import json, os, time, gc
import torch

if RUN_ROUND4_K_ONLY:
    print('[RUN_ROUND4_K_ONLY] Skipping Round 5 (K′ throughput).')
else:
    _ROUND_5_CACHE = os.path.join(ROUND_RESULTS_DIR, 'round_5_results.json')

    # Round 5 — K' throughput benchmarks (900s wall cap per leg). Round 4 K uses the same defaults as R2-optimized: manual batches + AMP + prefetch + fused ESMM (batch 4096).
    R5_K_PRIME_BATCH_SIZE = 4096
    R5_K_PRIME_BATCH_SIZE_LARGE = 8192
    R5_THROUGHPUT_AMP = True       # no effect without CUDA; BCE stays float32 after autocast forward
    R5_THROUGHPUT_PREFETCH = True  # row-group prep overlapped (ThreadPoolExecutor depth 1)
    R5_K_PRIME_MAX_WALL_SECONDS = 900  # ~15 min per train call; set None for no wall cap
    R5_K_PRIME_MAX_OPTIMIZER_STEPS = None
    R5_K_PRIME_MAX_BATCHES_PER_EPOCH = None
    R5_K_PRIME_MAX_ROW_GROUPS_PER_EPOCH = None

    # Round-5 extension: extra K' legs (torch.compile, pyarrow row groups). When cache exists, only missing keys run.
    R5_RUN_R3_LEGS = True


    def _round5_pack_train_meta(wall_outer, meta):
        row = {
            'wall_clock_seconds': int(wall_outer),
            'train_wall_seconds': int(round(meta['train_wall_seconds'])),
            'samples_per_sec': float(meta['samples_per_sec']),
            'samples_total_run': int(meta['samples_total_run']),
            'early_stop_reason': meta['early_stop_reason'],
            'use_amp': bool(meta.get('use_amp', False)),
            'prefetch_row_groups': bool(meta.get('prefetch_row_groups', False)),
        }
        mb = meta.get('use_manual_batches')
        if mb is not None:
            row['use_manual_batches'] = bool(mb)
        bs = meta.get('batch_size')
        if bs is not None:
            row['batch_size'] = int(bs)
        b = meta.get('cuda_max_memory_allocated_bytes')
        if b is not None:
            row['cuda_max_memory_allocated_bytes'] = int(b)
        for k in (
            'use_torch_compile',
            'torch_compile_active',
            'read_row_groups_as_arrow',
            'read_row_groups_arrow_used',
        ):
            if k in meta:
                row[k] = bool(meta[k])
        return row


    def _print_r5_row(name, m):
        if m is None:
            print(f'  {name}: OOM / skipped (null in JSON)')
            return
        esr = m.get('early_stop_reason')
        es_str = f", early_stop={esr!r}" if esr else ''
        amp_pf = ''
        if 'use_amp' in m:
            amp_pf = (
                f", amp={m.get('use_amp')}, prefetch={m.get('prefetch_row_groups')}, "
                f"manual={m.get('use_manual_batches')}, bs={m.get('batch_size')}"
            )
        tc = ''
        if m.get('use_torch_compile') is not None:
            tc = f", torch_compile={m.get('use_torch_compile')}, compile_active={m.get('torch_compile_active')}"
        arr = ''
        if m.get('read_row_groups_as_arrow') is not None:
            arr = (
                f", arrow_read={m.get('read_row_groups_as_arrow')}, "
                f"arrow_used={m.get('read_row_groups_arrow_used')}"
            )
        print(
            f"  {name}: wall={m['wall_clock_seconds']}s, samples/s={m['samples_per_sec']:,.0f}"
            f"{amp_pf}{tc}{arr}{es_str}"
        )


    results = {}
    if os.path.exists(_ROUND_5_CACHE):
        with open(_ROUND_5_CACHE) as f:
            results = json.load(f)

    need_full_r2 = not os.path.exists(_ROUND_5_CACHE)
    need_r3_compile = bool(R5_RUN_R3_LEGS) and ('K_prime_r3_compile' not in results)
    need_r3_pyarrow = bool(R5_RUN_R3_LEGS) and ('K_prime_r3_pyarrow' not in results)

    if not need_full_r2 and not need_r3_compile and not need_r3_pyarrow:
        print('ROUND 5: SKIPPED (cached; all configured legs present)')
        for name, m in results.items():
            if m is None:
                print(f'  {name}: (null — OOM or skipped)')
                continue
            _print_r5_row(name, m)
    else:
        # ===================== ROUND 5 (partial or full) =====================
        if need_full_r2:
            print('ROUND 5: running K_prime_r2_* legs (fresh cache)')
        else:
            print('ROUND 5: merging into existing round_5_results.json (R3 legs only)')

        t_prep0 = time.time()
        p_train, p_test = ensure_full_split_parquet_streaming(DATA_DIR, PROCESSED_FULL_DIR)
        if not (os.path.isfile(R4_NORM_TRAIN) and os.path.isfile(R4_NORM_TEST)):
            print('Streaming dense log1p -> normalized Parquet (Round 5 prep)...')
            stream_normalize_parquet(p_train, R4_NORM_TRAIN, SPARSE_COLS, DENSE_FEAT_COLS)
            stream_normalize_parquet(p_test, R4_NORM_TEST, SPARSE_COLS, DENSE_FEAT_COLS)
        print('Round 5: vocabs from cache or Parquet (same recipe / path as Round 4)...')
        vocabs_r5, cards_r5 = load_or_build_sparse_vocabs_filtered_parquet(
            p_train, SPARSE_COLS, min_count=5, cache_path=R4_FILTERED_VOCAB_CACHE,
            force_rebuild=FORCE_REBUILD_R4_VOCAB)
        print(f'  Prep done in {time.time() - t_prep0:.1f}s')

        _common_kw = dict(
            epochs=5,
            lr=1e-3,
            seed=RANDOM_STATE,
            max_wall_seconds=R5_K_PRIME_MAX_WALL_SECONDS,
            max_optimizer_steps=R5_K_PRIME_MAX_OPTIMIZER_STEPS,
            max_batches_per_epoch=R5_K_PRIME_MAX_BATCHES_PER_EPOCH,
            max_row_groups_per_epoch=R5_K_PRIME_MAX_ROW_GROUPS_PER_EPOCH,
        )

        _amp_on = bool(R5_THROUGHPUT_AMP and torch.cuda.is_available())
        _pref_on = bool(R5_THROUGHPUT_PREFETCH)

        if need_full_r2:
            # Leg A: baseline — DataLoader, FP32, no prefetch, batch 4096
            print('\n' + '=' * 60)
            print("K' leg A (baseline) — DataLoader, AMP off, prefetch off, batch=4096")
            print('=' * 60)
            t0 = time.perf_counter()
            model_ka, _, meta_ka = train_esmm_parquet_rowgroups(
                R4_NORM_TRAIN, vocabs_r5, cards_r5, SPARSE_COLS, DENSE_FEAT_COLS,
                batch_size=R5_K_PRIME_BATCH_SIZE,
                use_amp=False,
                prefetch_row_groups=False,
                use_manual_batches=False,
                **_common_kw,
            )
            wall_a = time.perf_counter() - t0
            results['K_prime_r2_baseline'] = _round5_pack_train_meta(wall_a, meta_ka)
            print(f"  >> K_prime_r2_baseline: wall={wall_a:.1f}s, samples/s={meta_ka['samples_per_sec']:,.0f}, "
                  f"early_stop={meta_ka['early_stop_reason']!r}")
            del model_ka
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            # Leg B: manual batches + fused ESMM (all ESMM) + AMP + prefetch, batch 4096
            print('\n' + '=' * 60)
            print(f"K' leg B (optimized) — manual batches, AMP={_amp_on}, prefetch={_pref_on}, batch=4096")
            print('=' * 60)
            t1 = time.perf_counter()
            model_kb, _, meta_kb = train_esmm_parquet_rowgroups(
                R4_NORM_TRAIN, vocabs_r5, cards_r5, SPARSE_COLS, DENSE_FEAT_COLS,
                batch_size=R5_K_PRIME_BATCH_SIZE,
                use_amp=_amp_on,
                prefetch_row_groups=_pref_on,
                use_manual_batches=True,
                **_common_kw,
            )
            wall_b = time.perf_counter() - t1
            results['K_prime_r2_optimized'] = _round5_pack_train_meta(wall_b, meta_kb)
            print(f"  >> K_prime_r2_optimized: wall={wall_b:.1f}s, samples/s={meta_kb['samples_per_sec']:,.0f}, "
                  f"early_stop={meta_kb['early_stop_reason']!r}")
            if torch.cuda.is_available() and meta_kb.get('cuda_max_memory_allocated_bytes'):
                print(f"  cuda_max_memory_allocated={meta_kb['cuda_max_memory_allocated_bytes'] / 1024**3:.2f} GiB (peak)")

            del model_kb
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            # Leg C: same as B but batch 8192; catch CUDA OOM for papermill stability
            print('\n' + '=' * 60)
            print(f"K' leg C (large batch) — manual batches, AMP={_amp_on}, prefetch={_pref_on}, batch=8192")
            print('=' * 60)
            results['K_prime_r2_batch8192'] = None
            try:
                t2 = time.perf_counter()
                model_kc, _, meta_kc = train_esmm_parquet_rowgroups(
                    R4_NORM_TRAIN, vocabs_r5, cards_r5, SPARSE_COLS, DENSE_FEAT_COLS,
                    batch_size=R5_K_PRIME_BATCH_SIZE_LARGE,
                    use_amp=_amp_on,
                    prefetch_row_groups=_pref_on,
                    use_manual_batches=True,
                    **_common_kw,
                )
                wall_c = time.perf_counter() - t2
                results['K_prime_r2_batch8192'] = _round5_pack_train_meta(wall_c, meta_kc)
                print(f"  >> K_prime_r2_batch8192: wall={wall_c:.1f}s, samples/s={meta_kc['samples_per_sec']:,.0f}, "
                      f"early_stop={meta_kc['early_stop_reason']!r}")
                del model_kc
            except RuntimeError as e:
                if 'out of memory' not in str(e).lower():
                    raise
                print(f'  K_prime_r2_batch8192: OOM (continuing; null in JSON): {e}')
                results['K_prime_r2_batch8192'] = None
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        def _run_r3_leg(key, use_torch_compile, read_arrow):
            print('\n' + '=' * 60)
            tag = 'torch.compile + batch8192 stack' if use_torch_compile else 'pyarrow row groups + batch8192 stack'
            print(f"K' R3 leg {key} — {tag}")
            print('=' * 60)
            results[key] = None
            try:
                t0 = time.perf_counter()
                model_m, _, meta_m = train_esmm_parquet_rowgroups(
                    R4_NORM_TRAIN, vocabs_r5, cards_r5, SPARSE_COLS, DENSE_FEAT_COLS,
                    batch_size=R5_K_PRIME_BATCH_SIZE_LARGE,
                    use_amp=_amp_on,
                    prefetch_row_groups=_pref_on,
                    use_manual_batches=True,
                    use_torch_compile=use_torch_compile,
                    read_row_groups_as_arrow=read_arrow,
                    **_common_kw,
                )
                wall_m = time.perf_counter() - t0
                results[key] = _round5_pack_train_meta(wall_m, meta_m)
                print(
                    f"  >> {key}: wall={wall_m:.1f}s, samples/s={meta_m['samples_per_sec']:,.0f}, "
                    f"early_stop={meta_m['early_stop_reason']!r}"
                )
                print(
                    f"  SCRIBE_R5_THROUGHPUT {key}: samples_per_sec={meta_m['samples_per_sec']:.2f} "
                    f"train_wall_s={meta_m['train_wall_seconds']:.1f} samples={meta_m['samples_total_run']}"
                )
                del model_m
            except RuntimeError as e:
                if 'out of memory' not in str(e).lower():
                    raise
                print(f'  {key}: OOM (null in JSON): {e}')
                results[key] = None
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        if need_r3_compile:
            _run_r3_leg('K_prime_r3_compile', use_torch_compile=True, read_arrow=False)
        if need_r3_pyarrow:
            _run_r3_leg('K_prime_r3_pyarrow', use_torch_compile=False, read_arrow=True)

        with open(_ROUND_5_CACHE, 'w') as f:
            json.dump(results, f)

        print('\n' + '=' * 60)
        print("ROUND 5 SUMMARY (K' legs)")
        print('=' * 60)
        for name, m in results.items():
            _print_r5_row(name, m)

        if R5_RUN_R3_LEGS:
            print('\n--- R3 extension (SCRIBE throughput one-liners) ---')
            for key in ('K_prime_r3_compile', 'K_prime_r3_pyarrow'):
                m = results.get(key)
                if isinstance(m, dict) and m.get('samples_per_sec') is not None:
                    print(
                        f"  SCRIBE_R5_THROUGHPUT {key}: samples_per_sec={m['samples_per_sec']:.2f} "
                        f"wall_clock_s={m['wall_clock_seconds']} train_wall_s={m['train_wall_seconds']:.1f}"
                    )


In [ ]:

import json, os, time, gc
import torch

if RUN_ROUND4_K_ONLY:
    print('[RUN_ROUND4_K_ONLY] Skipping Round 6 (architecture shootout).')
else:
    _ROUND_6_CACHE = os.path.join(ROUND_RESULTS_DIR, 'round_6_results.json')
    _ROUND_4_MARKER = os.path.join(ROUND_RESULTS_DIR, 'round_4_results.json')

    # Ignore K_EARLY_STOP_* here: if those caps are set for dev runs of Experiment K, Round 6
    # would train on a fraction of an epoch and K_ref metrics collapse (~0.5 AUC). MTL shootout = full 5 epochs.
    _R6_TRAIN_KW = dict(
        epochs=5,
        batch_size=4096,
        lr=1e-3,
        seed=RANDOM_STATE,
        max_wall_seconds=None,
        max_optimizer_steps=None,
        max_batches_per_epoch=None,
        max_row_groups_per_epoch=None,
    )


    def _r6_float_fmt(x):
        if x is None or (isinstance(x, float) and (x != x)):
            return 'nan'
        return f'{float(x):.4f}'


    if not os.path.exists(_ROUND_6_CACHE):
        # ===================== ROUND 6 =====================
        _ok_r4 = os.path.isfile(_ROUND_4_MARKER)
        _ok_train_pq = os.path.isfile(R4_NORM_TRAIN)
        _ok_test_pq = os.path.isfile(R4_NORM_TEST)
        if not (_ok_r4 and _ok_train_pq and _ok_test_pq):
            print(
                'ROUND 6: SKIPPED — need Round 4 results cache and normalized Parquet. '
                f'round_4_results={_ok_r4}, R4_NORM_TRAIN={_ok_train_pq}, R4_NORM_TEST={_ok_test_pq}. '
                'Run Round 4 first (Parquet + vocabs).'
            )
        else:
            results = {}
            print('ROUND 6: loading full-split paths + filtered vocabs (same as Experiment K / Round 4)...')
            p_train, _p_test_paths = ensure_full_split_parquet_streaming(DATA_DIR, PROCESSED_FULL_DIR)
            vocabs_r6, cards_r6 = load_or_build_sparse_vocabs_filtered_parquet(
                p_train, SPARSE_COLS, min_count=5, cache_path=R4_FILTERED_VOCAB_CACHE,
                force_rebuild=FORCE_REBUILD_R4_VOCAB,
            )

            legs = [
                (
                    'K_ref',
                    None,
                    {},
                    'ESMM baseline (same as K)',
                ),
                (
                    'SharedBottom',
                    ESMM_SharedBottom,
                    {'trunk_dims': (360, 200, 80)},
                    'Shared trunk 360->200->80 + two heads',
                ),
                (
                    'MMoE',
                    ESMM_MMoE,
                    {'num_experts': 4, 'expert_hidden': 360, 'd_model': 128, 'tower_hidden_ratio': 0.5},
                    'MMoE E=4, expert_hidden=360, d_model=128',
                ),
                (
                    'PLE',
                    ESMM_PLE,
                    {
                        'd_model': 128,
                        'expert_hidden': 256,
                        'num_shared_experts': 1,
                        'num_task_experts': 1,
                        'dropout': 0.0,
                    },
                    '2-level PLE, 1 shared + 1 task expert per side',
                ),
            ]

            _r6_skip = {
                'K_ref': SKIP_ROUND6_K_REF,
                'SharedBottom': SKIP_ROUND6_SHARED_BOTTOM,
                'MMoE': SKIP_ROUND6_MMOE,
                'PLE': SKIP_ROUND6_PLE,
            }
            legs = [leg for leg in legs if not _r6_skip.get(leg[0], False)]
            if not legs:
                print('ROUND 6: SKIPPED — all legs disabled by SKIP_ROUND6_* flags.')
            else:
                print(f'ROUND 6: running {len(legs)} leg(s): {[leg[0] for leg in legs]}')

            if legs:
                for key, ctor, ctor_kw, desc in legs:
                    print('\n' + '=' * 60)
                    print(f'ROUND 6 — {key}: {desc}')
                    print('=' * 60)
                    t0 = time.time()
                    model_r6, _, train_meta = train_esmm_parquet_rowgroups(
                        R4_NORM_TRAIN, vocabs_r6, cards_r6, SPARSE_COLS, DENSE_FEAT_COLS,
                        model_ctor=ctor,
                        model_ctor_kwargs=ctor_kw if ctor_kw else None,
                        **_R6_TRAIN_KW,
                    )
                    wall = time.time() - t0
                    n_params = int(sum(p.numel() for p in model_r6.parameters()))
                    print(
                        f"  Train: wall={wall:.0f}s, throughput={train_meta['samples_per_sec']:,.0f} samples/s, "
                        f"early_stop={train_meta['early_stop_reason']!r}, params={n_params:,}"
                    )
                    metrics = evaluate_esmm_multitask_streaming_parquet(
                        model_r6, R4_NORM_TEST, vocabs_r6, SPARSE_COLS, DENSE_FEAT_COLS,
                    )
                    # Same eval path as Round 4 Experiment K (sanity vs multitask helper).
                    ctcvr_legacy, _ = evaluate_esmm_ctcvr_streaming_parquet(
                        model_r6, R4_NORM_TEST, vocabs_r6, SPARSE_COLS, DENSE_FEAT_COLS,
                    )
                    cvr_legacy, _ = evaluate_esmm_cvr_streaming_parquet(
                        model_r6, R4_NORM_TEST, vocabs_r6, SPARSE_COLS, DENSE_FEAT_COLS,
                    )
                    metrics['CTCVR_AUC_legacy_r4'] = float(ctcvr_legacy)
                    metrics['CVR_AUC_legacy_r4'] = float(cvr_legacy)
                    for mk, mv in metrics.items():
                        print(f'  {mk}: {_r6_float_fmt(mv)}')
                    results[key] = {
                        **metrics,
                        'wall_clock_seconds': int(wall),
                        'num_parameters': n_params,
                        'train_samples_per_sec': float(train_meta['samples_per_sec']),
                        'train_wall_seconds': float(train_meta['train_wall_seconds']),
                        'early_stop_reason': train_meta['early_stop_reason'],
                    }
                    del model_r6
                    gc.collect()
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()

                with open(_ROUND_6_CACHE, 'w') as f:
                    json.dump(results, f)

            if results:
                print('\n' + '=' * 60)
                print('ROUND 6 SUMMARY (architecture comparison)')
                print('=' * 60)
                print(
                    'key  CTCVR_AUC  CTCVR_PR  CTR_AUC  CTR_PR  ll_ctr  ll_ctcvr  ECE_ctr  ECE_ctcvr  CVR_AUC  params  wall_s'
                )
                for key, m in results.items():
                    print(
                        f"{key:12s} {_r6_float_fmt(m.get('CTCVR_AUC')):>9} {_r6_float_fmt(m.get('CTCVR_PR_AUC')):>8} "
                        f"{_r6_float_fmt(m.get('CTR_AUC')):>7} {_r6_float_fmt(m.get('CTR_PR_AUC')):>7} "
                        f"{_r6_float_fmt(m.get('logloss_ctr')):>7} {_r6_float_fmt(m.get('logloss_ctcvr')):>9} "
                        f"{_r6_float_fmt(m.get('ECE_ctr')):>8} {_r6_float_fmt(m.get('ECE_ctcvr')):>10} "
                        f"{_r6_float_fmt(m.get('CVR_AUC')):>8} {m.get('num_parameters', 0):>8,} {m.get('wall_clock_seconds', 0):>7}"
                    )
    else:
        with open(_ROUND_6_CACHE) as f:
            results = json.load(f)
        print('ROUND 6: SKIPPED (cached)')
        for key, m in results.items():
            print(
                f"  {key}: CTCVR_AUC={_r6_float_fmt(m.get('CTCVR_AUC'))}, CTR_AUC={_r6_float_fmt(m.get('CTR_AUC'))}, "
                f"CVR_AUC={_r6_float_fmt(m.get('CVR_AUC'))}, params={m.get('num_parameters', 0):,}"
            )
